In [ ]:
# !pip install adjustText

from __future__ import annotations

from pathlib import Path
import os
import warnings
import requests
import seaborn as sns

import pandas as pd
import numpy as np
import re

import geopandas as gpd
from pyproj import Transformer
from shapely.geometry import Point
import folium

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

from typing import Optional, Tuple, Dict
from datetime import datetime
from matplotlib.ticker import FuncFormatter
from adjustText import adjust_text
from matplotlib.ticker import PercentFormatter


warnings.filterwarnings("ignore")
pd.options.display.float_format = '{:.2f}'.format

# 한글 및 마이너스 깨짐 방지
try:
    mpl.rcParams["font.family"] = "Malgun Gothic"   # Windows
except:
    mpl.rcParams["font.family"] = "AppleGothic"    # Mac

mpl.rcParams["axes.unicode_minus"] = False

In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import warnings
import requests
from pathlib import Path

import pandas as pd
import numpy as np
import re
import seaborn as sns

import geopandas as gpd
from pyproj import Transformer
from shapely.geometry import Point

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

from typing import Optional, Tuple, Dict
from datetime import datetime
import folium

warnings.filterwarnings("ignore")
pd.options.display.float_format = '{:.2f}'.format

# Mac용 한글 폰트(AppleGothic) 및 마이너스 깨짐 방지
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

%config InlineBackend.figure_format = 'retina'

# 행정동 경계

In [ ]:
BASE_DIR = os.getcwd()                     # 현재 작업 중인 기본 폴더 경로
FOLDER_DIR = os.path.join(BASE_DIR, "완전최종_전처리완료_GPKG모음")  # 가공 데이터 저장 폴더

DATA_DIR = os.path.join(FOLDER_DIR, "서울시_행정동_EPSG5179.gpkg")

gdf_dong = gpd.read_file(DATA_DIR)

gdf_dong

In [ ]:
gdf_dong.plot(figsize=(8, 8), edgecolor="black")
plt.title("서울시 행정동 경계")
plt.axis("off")
plt.show()

# 자치구 경계

In [ ]:
BASE_DIR = os.getcwd()                     # 현재 작업 중인 기본 폴더 경로
FOLDER_DIR = os.path.join(BASE_DIR, "완전최종_전처리완료_GPKG모음")  # 가공 데이터 저장 폴더

DATA_DIR = os.path.join(FOLDER_DIR, "서울시_자치구_EPSG5179.gpkg")

gdf_SIGNGU = gpd.read_file(DATA_DIR)

gdf_SIGNGU

In [ ]:
gdf_SIGNGU.plot(figsize=(8, 8), edgecolor="black")
plt.title("서울시 자치구 경계")
plt.axis("off")
plt.show()

# 자치구별 방문자수 추이

In [ ]:
BASE_DIR = os.getcwd()                     # 현재 작업 중인 기본 폴더 경로
FOLDER_DIR = os.path.join(BASE_DIR, "완전최종_전처리완료_CSV모음")  # 가공 데이터 저장 폴더

DATA_DIR = os.path.join(FOLDER_DIR, "자치구별_내외국인_방문자_수_추이.csv")

visiter = pd.read_csv(DATA_DIR)

In [ ]:
visiter

In [ ]:
visiter["현지인방문자수"] = pd.to_numeric(visiter["현지인방문자수"]) # 남자 인구 컬럼을 숫자형으로 변환 (문자·기호는 NaN 처리)
visiter["외지인방문자수"] = pd.to_numeric(visiter["외지인방문자수"]) # 여자 인구 컬럼을 숫자형으로 변환
visiter["외국인방문자수"] = pd.to_numeric(visiter["외국인방문자수"]) # 여자 인구 컬럼을 숫자형으로 변환
visiter["총방문자수"] = visiter["현지인방문자수"] + visiter["외지인방문자수"] + visiter["외국인방문자수"] # 현지인 + 외지인 + 외국인을 더해 총방문자수 컬럼 생성
visiter_df = visiter
visiter_df.head()

## 자치구 경계와 결합

In [ ]:
SIGNGU_nm_to_id = gdf_SIGNGU.set_index("SIGNGU_NM")["SIGNGU_CD"].to_dict() # 행정동 이름(region_nm)을 키로, 행정동 코드(region_id)를 값으로 하는 매핑 딕셔너리 생성

visiter_df["SIGNGU_CD"] = visiter_df["자치구"].map(SIGNGU_nm_to_id) # 등록인구 데이터의 행정동 이름을 기준으로 행정동 코드(region_id)를 매칭하여 추가
visiter_df.head()

In [ ]:
def norm_nm(x):
    if pd.isna(x):
        return None

    s = str(x).strip()
    s = s.replace("\u00A0", " ")
    s = " ".join(s.split())

    # '.' ↔ '·' 통일 (종로1.2.3.4가동 → 종로1·2·3·4가동)
    s = s.replace(".", "·")

    return s

In [ ]:
# 자치구 이름을 표준화 함수(norm_nm)로 정리(공백·특수문자·표기 차이를 통일하여 이후 데이터 매칭 정확도 향상)

gdf_SIGNGU["SIGNGU_NM"] = gdf_SIGNGU["SIGNGU_NM"].map(norm_nm)

In [ ]:
def ensure_crs(gdf, epsg):
    # GeoDataFrame에 좌표계(CRS)가 설정되어 있는지 확인
    if gdf.crs is None:
        raise ValueError("행정동 경계 레이어 CRS 없음")

    # 지정한 EPSG 좌표계로 변환하여 반환
    return gdf.to_crs(epsg=epsg)

In [ ]:
gdf_SIGNGU = ensure_crs(gdf_SIGNGU, 5179)

In [ ]:
SIGNGU_area_df = gdf_SIGNGU[["SIGNGU_CD", "geometry"]].copy() # 자치구 이름(region_nm)을 함께 추출하여 면적 테이블 생성

SIGNGU_area_df["area_km2"] = gdf_SIGNGU.geometry.area / 1e6 # 각 자치구 폴리곤의 면적을 계산한 뒤 m² → km² 단위로 변환하여 저장

In [ ]:
SIGNGU_area_df

## 자치구별 방문자수 밀도

In [ ]:
visiter_df

In [ ]:
# 면적 데이터와 등록인구 데이터를 행정동 코드(region_id) 기준으로 병합 각 행정동에 해당하는 면적(km²)을 추가
visiter_df = visiter_df.merge(SIGNGU_area_df[["SIGNGU_CD", "area_km2"]], on="SIGNGU_CD", how="left")

# 인구밀도 계산 (1 km²당 인구 수) - 총인구 ÷ 면적 → 인구밀도 컬럼 생성
visiter_df["pop_density"] = visiter_df["총방문자수"] / visiter_df["area_km2"]
visiter_df.head()

In [ ]:
visiter_df

In [ ]:
# 자치구 공간데이터(gdf_admin)와 인구밀도 테이블(pop_df)을 region_id 기준으로 병합
gdf_map = gdf_SIGNGU.merge(
    visiter_df[["SIGNGU_CD", "pop_density"]],
    on="SIGNGU_CD",
    how="left"
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

gdf_map.plot(
    column="pop_density",      # 색칠 기준 컬럼 (인구밀도)
    cmap="OrRd",               # 색상 팔레트 (연한색 → 진한색)
    legend=True,               # 범례 표시
    edgecolor="black",
    linewidth=0.2,
    ax=ax
)

plt.title("서울시 자치구 방문자수 밀도 단계구분도", fontsize=14)
plt.axis("off")
plt.show()

In [ ]:
visiter_df.sort_values(by="pop_density", ascending=False).reset_index(drop=True)

In [ ]:
SIGNGU_visiter_df = visiter_df.groupby('자치구')[['총방문자수', '외지인방문자수', '외국인방문자수', 'area_km2']].sum()

In [ ]:
SIGNGU_visiter_df

In [ ]:
# 인구밀도 계산 (1 km²당 인구 수) - 총인구 ÷ 면적 → 인구밀도 컬럼 생성
SIGNGU_visiter_df["pop_density"] = SIGNGU_visiter_df["총방문자수"] / SIGNGU_visiter_df["area_km2"]
SIGNGU_visiter_df.head()

In [ ]:
SIGNGU_visiter_df["pop_density2"] = SIGNGU_visiter_df["외지인방문자수"] / SIGNGU_visiter_df["area_km2"]
SIGNGU_visiter_df.head()

In [ ]:
SIGNGU_visiter_df["pop_density3"] = SIGNGU_visiter_df["외국인방문자수"] / SIGNGU_visiter_df["area_km2"]
SIGNGU_visiter_df.head()

In [ ]:
SIGNGU_visiter_df["pop_density4"] = (SIGNGU_visiter_df["외지인방문자수"] + SIGNGU_visiter_df["외국인방문자수"]) / SIGNGU_visiter_df["area_km2"]
SIGNGU_visiter_df.head()

In [ ]:
# 자치구별 총합 집계 및 정렬
SIGNGU_total = SIGNGU_visiter_df.groupby('자치구')['pop_density'].sum().sort_values(ascending=False)

# 상위 10개 구 시각화
plt.figure(figsize=(12, 6))
sns.barplot(x=SIGNGU_total.head(10).index, y=SIGNGU_total.head(10).values)
plt.title('2025년 총 방문자 수 밀도 Top 10 자치구')
plt.show()

In [ ]:
# 자치구별 총합 집계 및 정렬
SIGNGU_total = SIGNGU_visiter_df.groupby('자치구')['pop_density2'].sum().sort_values(ascending=False)

# 상위 10개 구 시각화
plt.figure(figsize=(12, 6))
sns.barplot(x=SIGNGU_total.head(10).index, y=SIGNGU_total.head(10).values)
plt.title('2025년 외지인 방문자 수 밀도 Top 10 자치구')
plt.show()

In [ ]:
# 자치구별 총합 집계 및 정렬
SIGNGU_total = SIGNGU_visiter_df.groupby('자치구')['pop_density3'].sum().sort_values(ascending=False)

# 상위 10개 구 시각화
plt.figure(figsize=(12, 6))
sns.barplot(x=SIGNGU_total.head(10).index, y=SIGNGU_total.head(10).values)
plt.title('2025년 외국인 방문자 수 밀도 Top 10 자치구')
plt.show()

In [ ]:
# 자치구별 총합 집계 및 정렬
SIGNGU_total = SIGNGU_visiter_df.groupby('자치구')['pop_density4'].sum().sort_values(ascending=False)

# 상위 10개 구 시각화
plt.figure(figsize=(12, 6))
sns.barplot(x=SIGNGU_total.head(10).index, y=SIGNGU_total.head(10).values)
plt.title('2025년 외지인+외국인 방문자 수 밀도 Top 10 자치구')
plt.show()

## 방문자수 합계

In [ ]:
# 분기별, 자치구별 방문자 합계 요약
quarterly_summary = visiter.groupby(['분기', '자치구'])[['현지인방문자수', '외지인방문자수', '외국인방문자수']].sum()

# 만약 자치구 구분 없이 분기 전체 합계만 보고 싶다면:
# quarterly_summary = native_reset.groupby('분기')[['현지인방문자', '외지인방문자', '전체방문자']].sum()

quarterly_summary

In [ ]:
month_summary = visiter.groupby('월')[['현지인방문자수', '외지인방문자수', '외국인방문자수']].sum()

In [ ]:
month_summary

In [ ]:
visiter.groupby('자치구')[['현지인방문자수', '외지인방문자수', '외국인방문자수']].sum().plot.line(color=['gray', 'purple', 'red'])

In [ ]:
whole_summary = visiter.groupby('분기')[['현지인방문자수', '외지인방문자수', '외국인방문자수']].sum()

whole_summary

In [ ]:
visiter.groupby('분기')[['현지인방문자수', '외지인방문자수', '외국인방문자수']].sum().plot.bar(color=['gray', 'purple', 'red'])

In [ ]:
# 한글 폰트 설정 (윈도우 맑은 고딕 예시, Mac은 'AppleGothic')
plt.rc('font', family='Malgun Gothic')

# 총방문자수 컬럼 추가
visiter['총방문자수'] = visiter['현지인방문자수'] + visiter['외지인방문자수'] + visiter['외국인방문자수']
visiter['외지인+외국인'] = visiter['외지인방문자수'] + visiter['외국인방문자수']

# 기본 통계량 확인
print(visiter.describe())

In [ ]:
visiter

In [ ]:
# 월별 방문자 수 집계
monthly_trend = visiter.groupby('월')[['현지인방문자수', '외지인방문자수', '외국인방문자수', '외지인+외국인']].sum()

# 선 그래프 시각화
monthly_trend.plot(kind='line', marker='o', figsize=(10, 6))
plt.title('2025년 월별 서울시 방문자 수 추이')
plt.xlabel('월')
plt.ylabel('방문자 수')
plt.grid(True)
plt.show()

In [ ]:
# 자치구별 총합 집계 및 정렬
district_total = visiter.groupby('자치구')['총방문자수'].sum().sort_values(ascending=False)

# 상위 10개 구 시각화
plt.figure(figsize=(12, 6))
sns.barplot(x=district_total.head(10).index, y=district_total.head(10).values)
plt.title('2025년 총 방문자 수 Top 10 자치구')
plt.show()

In [ ]:
# 자치구별 총합 집계 및 정렬
district_total = visiter.groupby('자치구')['외지인방문자수'].sum().sort_values(ascending=False)

# 상위 10개 구 시각화
plt.figure(figsize=(12, 6))
sns.barplot(x=district_total.head(10).index, y=district_total.head(10).values)
plt.title('2025년 외지인 방문자 수 Top 10 자치구')
plt.show()

In [ ]:
# 자치구별 총합 집계 및 정렬
district_total = visiter.groupby('자치구')['외국인방문자수'].sum().sort_values(ascending=False)

# 상위 10개 구 시각화
plt.figure(figsize=(12, 6))
sns.barplot(x=district_total.head(10).index, y=district_total.head(10).values)
plt.title('2025년 외국인 방문자 수 Top 10 자치구')
plt.show()

In [ ]:
# 자치구별 총합 집계 및 정렬
district_total = visiter.groupby('자치구')['외지인+외국인'].sum().sort_values(ascending=False)

# 상위 10개 구 시각화
plt.figure(figsize=(12, 6))
sns.barplot(x=district_total.head(10).index, y=district_total.head(10).values)
plt.title('2025년 외지인+외국인 방문자 수 Top 10 자치구')
plt.show()

In [ ]:
visiter.groupby('자치구')['외지인방문자수'].sum().sort_values(ascending=False)

In [ ]:
# 자치구별 유형별 합계
district_composition = visiter.groupby('자치구')[['현지인방문자수', '외지인방문자수', '외국인방문자수']].sum()

# 비율(%)로 변환
district_ratio = district_composition.div(district_composition.sum(axis=1), axis=0) * 100

# 누적 막대 그래프 (Stacked Bar Chart)
district_ratio.plot(kind='bar', stacked=True, figsize=(14, 7), colormap='Set2')
plt.title('자치구별 방문자 유형 비율(%)')
plt.ylabel('비율(%)')
plt.legend(loc='upper right', bbox_to_anchor=(1.15, 1))
plt.show()

In [ ]:
# 한글 폰트 설정 (환경에 맞게 수정)
plt.rc('font', family='Malgun Gothic')

# --- 1-1. 상관계수 히트맵 (Heatmap) ---
# 현지인, 외지인, 외국인 방문자 수 간의 피어슨 상관계수 계산
corr_matrix = visiter[['현지인방문자수', '외지인방문자수', '외국인방문자수']].corr()

plt.figure(figsize=(8, 6))
# annot=True: 박스 안에 수치 표시, cmap: 색상 테마
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title('방문자 유형 간 상관관계 히트맵')
plt.show()

# --- 1-2. 산점도 (Scatter Plot) ---
# 외지인과 외국인 방문자 수가 비례하는지 시각적으로 확인
plt.figure(figsize=(10, 6))
# hue='자치구'를 넣으면 자치구별로 색상이 다르게 표시됩니다. (범례가 너무 많으면 legend=False로 끕니다)
sns.scatterplot(data=visiter, x='외지인방문자수', y='외국인방문자수', alpha=0.7)
plt.title('외지인 vs 외국인 방문자 수 관계')
plt.xlabel('외지인 방문자 수')
plt.ylabel('외국인 방문자 수')
plt.grid(True)
plt.show()

In [ ]:
# 총방문자수 컬럼이 없다면 생성해 줍니다.
if '총방문자수' not in visiter.columns:
    visiter['총방문자수'] = visiter['현지인방문자수'] + visiter['외지인방문자수'] + visiter['외국인방문자수']

plt.figure(figsize=(10, 6))
# x축을 분기, y축을 총방문자수로 설정하여 박스 플롯 생성
sns.boxplot(data=visiter, x='분기', y='총방문자수', order=['Q1', 'Q2', 'Q3', 'Q4'], palette='pastel')

plt.title('분기별 서울시 자치구 총 방문자 수 분포')
plt.xlabel('분기')
plt.ylabel('총 방문자 수')
plt.show()

# 숙박 체류시간

In [ ]:
BASE_DIR = os.getcwd()                     # 현재 작업 중인 기본 폴더 경로
FOLDER_DIR = os.path.join(BASE_DIR, "완전최종_전처리완료_CSV모음")  # 가공 데이터 저장 폴더

DATA_DIR = os.path.join(FOLDER_DIR, "자치구별_숙박방문외지인_숙박_체류시간_월별데이터.csv")

stay = pd.read_csv(DATA_DIR)

In [ ]:
stay

In [ ]:
# --- 1-1. 숙박방문자수 Top 10 자치구 (2025년 총합 기준) ---
# 자치구별로 숙박방문자수를 모두 더한 후 내림차순 정렬
top10_visitors = stay.groupby('자치구')['숙박방문자수'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top10_visitors.index, y=top10_visitors.values, palette='viridis')
plt.title('2025년 숙박방문자수 Top 10 자치구')
plt.ylabel('총 숙박방문자수')
plt.show()

# --- 1-2. 평균 숙박일수 vs 평균 체류시간 산점도 ---
# 자치구별 평균 숙박일수와 체류시간 계산
district_avg = stay.groupby('자치구')[['평균 숙박일수', '평균체류시간(시간)']].mean()

plt.figure(figsize=(10, 6))
sns.scatterplot(data=district_avg, x='평균 숙박일수', y='평균체류시간(시간)', s=100, color='coral')

# 각 점 옆에 자치구 이름 표시 (어떤 구인지 직관적으로 확인하기 위함)
for idx, row in district_avg.iterrows():
    plt.text(row['평균 숙박일수'] + 0.01, row['평균체류시간(시간)'], idx, fontsize=9)

plt.title('자치구별 평균 숙박일수 vs 평균 체류시간')
plt.xlabel('평균 숙박일수')
plt.ylabel('평균 체류시간(시간)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# '체류시간_Gap' 파생 변수 생성: 지역 평균 - 전국 평균
stay['체류시간_Gap'] = stay['평균체류시간(시간)'] - stay['전국 평균체류시간(시간)']

# 자치구별 Gap의 평균을 구하고 오름차순 정렬
gap_avg = stay.groupby('자치구')['체류시간_Gap'].mean().sort_values()

plt.figure(figsize=(10, 8))
# Gap이 0보다 작으면 빨간색 계열, 크면 파란색 계열로 색상 지정
colors = ['tomato' if x < 0 else 'cornflowerblue' for x in gap_avg.values]

# 가로 막대 그래프로 표현 (보기 편하게)
sns.barplot(x=gap_avg.values, y=gap_avg.index, palette=colors)

plt.title('자치구별 전국 평균 대비 체류시간 갭 (지역평균 - 전국평균)')
plt.xlabel('체류시간 갭 (시간)')
plt.axvline(0, color='black', linewidth=1) # 0 기준선 추가
plt.show()

In [ ]:
# --- 3-1. 월별 숙박방문자수 추이 (선 그래프) ---
monthly_visitors = stay.groupby('월')['숙박방문자수'].sum()

plt.figure(figsize=(10, 5))
monthly_visitors.plot(kind='line', marker='o', color='teal', linewidth=2)
plt.title('2025년 월별 서울시 총 숙박방문자수 추이')
plt.xlabel('월')
plt.ylabel('숙박방문자수')
plt.xticks(range(1, 13)) # x축 눈금을 1~12월로 고정
plt.grid(True)
plt.show()

# --- 3-2. 분기별 평균 체류시간 분포 (박스 플롯) ---
plt.figure(figsize=(10, 6))
sns.boxplot(data=stay, x='분기', y='평균체류시간(시간)', order=['Q1', 'Q2', 'Q3', 'Q4'], palette='Set3')
plt.title('분기별 자치구 평균 체류시간 분포')
plt.xlabel('분기')
plt.ylabel('평균 체류시간 (시간)')
plt.show()

In [ ]:
# 상관계수를 확인할 주요 수치형 컬럼만 선택
corr_cols = ['숙박방문자수', '평균 숙박일수', '평균체류시간(시간)', '전국 평균체류시간(시간)']
corr_matrix2 = stay[corr_cols].corr()

plt.figure(figsize=(8, 6))
# cmap='RdYlBu_r'로 양의 상관관계는 파란색, 음은 빨간색으로 표시
sns.heatmap(corr_matrix2, annot=True, cmap='RdYlBu_r', fmt=".2f", vmin=-1, vmax=1, linewidths=0.5)
plt.title('숙박 및 체류 지표 간 상관관계 히트맵')
plt.show()

# 관광숙박업 인허가 정보

In [ ]:
BASE_DIR = os.getcwd()                     # 현재 작업 중인 기본 폴더 경로
FOLDER_DIR = os.path.join(BASE_DIR, "완전최종_전처리완료_CSV모음")  # 가공 데이터 저장 폴더

DATA_DIR = os.path.join(FOLDER_DIR, "서울시_관광숙박업_인허가_정보.csv")

hotel = pd.read_csv(DATA_DIR)

In [ ]:
hotel

In [ ]:
# '서울특별시 서초구 바우뫼로...' 에서 두 번째 단어(서초구) 추출
hotel['자치구'] = hotel['도로명전체주소'].dropna().apply(lambda x: x.split()[1] if len(x.split()) > 1 else None)

# 추출이 잘 되었는지 확인
print(hotel['자치구'].unique())

In [ ]:
plt.rc('font', family='Malgun Gothic')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 숙박업 상세명 빈도 막대 그래프
sns.countplot(data=hotel, y='관광숙박업상세명', order=hotel['관광숙박업상세명'].value_counts().index, ax=axes[0], palette='pastel')
axes[0].set_title('숙박업 유형별 개수')

# 상세 영업 상태 막대 그래프
sns.countplot(data=hotel, x='상세영업상태명', order=hotel['상세영업상태명'].value_counts().index, ax=axes[1], palette='Set2')
axes[1].set_title('영업 상태별 개수')

plt.tight_layout()
plt.show()

In [ ]:
# 현재 '영업중'인 호텔만 필터링 (예시: 영업 상태명이 '영업중'인 경우)
# 실제 데이터의 '영업' 상태명 텍스트에 맞게 수정하세요.
active_hotels = hotel[hotel['상세영업상태명'] == '영업중']

plt.figure(figsize=(10, 8))
# X좌표(경도), Y좌표(위도)를 이용해 산점도 시각화
sns.scatterplot(data=active_hotels, x='X좌표(WGS84)', y='Y좌표(WGS84)',
                hue='관광숙박업상세명', alpha=0.6, s=50)

plt.title('서울시 영업 중인 관광숙박업소 위치 분포')
plt.xlabel('경도 (Longitude)')
plt.ylabel('위도 (Latitude)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
# 1. 서울 중심 좌표 설정 (위도: Y좌표, 경도: X좌표)
seoul_center = [37.5665, 126.9780]

# 2. Folium 지도 객체 생성 (zoom_start로 초기 확대 정도 설정)
m = folium.Map(location=seoul_center, zoom_start=11, tiles='CartoDB positron')
# tiles='CartoDB positron'은 지도를 깔끔한 밝은 톤으로 만들어주어 데이터(점)가 잘 보입니다.

# 3. 숙박업 유형별 마커 색상 지정 (Seaborn의 hue 역할)
# 데이터에 있는 유형들에 맞춰 원하는 색상으로 매핑합니다.
color_dict = {
    '관광호텔업': 'blue',
    '호스텔업': 'orange',
    '가족호텔업': 'green',
    '소형호텔업': 'red',
    '한국전통호텔업': 'purple',
    '의료관광호텔업': 'brown',
    '휴양콘도미니엄업': 'pink'
}

# 4. 좌표 데이터가 없는 결측치(NaN) 행 제거 (에러 방지)
active_hotels_clean = active_hotels.dropna(subset=['Y좌표(WGS84)', 'X좌표(WGS84)'])

# 5. 데이터를 반복하며 지도에 점(CircleMarker) 찍기
for idx, row in active_hotels_clean.iterrows():
    hotel_type = row['관광숙박업상세명']

    # 딕셔너리에서 색상을 찾고, 매핑되지 않은 유형은 'gray'로 처리
    marker_color = color_dict.get(hotel_type, 'gray')

    # 마커 클릭 시 보여줄 정보 (사업장명과 숙박업 유형)
    popup_text = f"<b>{row['사업장명']}</b><br>{hotel_type}"

    folium.CircleMarker(
        # folium은 반드시 [위도(Y), 경도(X)] 순서로 입력해야 합니다!
        location=[row['Y좌표(WGS84)'], row['X좌표(WGS84)']],
        radius=4,               # 점의 크기
        color=marker_color,     # 테두리 색상
        fill=True,              # 안쪽 채우기
        fill_color=marker_color,# 안쪽 색상
        fill_opacity=0.7,       # 투명도
        popup=folium.Popup(popup_text, max_width=250) # 클릭 시 팝업창
    ).add_to(m)

# 6. 지도 출력 (주피터 노트북 환경에서는 m을 입력하면 지도가 바로 뜹니다)
m

In [ ]:
# 1. 서울 중심 좌표 설정 (이전과 동일)
seoul_center = [37.5665, 126.9780]

# 2. Folium 지도 객체 생성 (이전과 동일한 지도 스타일)
m = folium.Map(location=seoul_center, zoom_start=11)
# 참고: 마커가 훨씬 더 잘 보이게 하려면 지도 배경을 단순한 흰색 계열로 바꾸는 것도 좋습니다.
# m = folium.Map(location=seoul_center, zoom_start=11, tiles='CartoDB positron')

# 3. 영업 중인 호텔만 지도에 마커 추가
# 좌표 데이터가 없는 결측치(NaN) 행 제거
active_hotels_clean = active_hotels.dropna(subset=['Y좌표(WGS84)', 'X좌표(WGS84)'])

for idx, row in active_hotels_clean.iterrows():
    folium.CircleMarker(
        # [위도, 경도] 순서 중요!
        location=[row['Y좌표(WGS84)'], row['X좌표(WGS84)']],

        # --- 여기부터 마커 스타일 수정 ---
        radius=5,              # 마커 크기 확대 (기존 3 -> 5)
        color='darkblue',      # 테두리 색상을 더 진한 파란색으로 변경
        weight=2,              # 테두리 선 두께 증가
        fill=True,
        fill_color='dodgerblue', # 내부 색상을 더 밝고 선명한 파란색으로
        fill_opacity=0.8,      # 투명도 조절 (기존 0.6 -> 0.8, 덜 투명하게)
        # --------------------------------

        popup=row['사업장명']
    ).add_to(m)

# 4. 지도 출력 (주피터 노트북 환경)
m

In [ ]:
# 한글 폰트 및 마이너스 깨짐 방지 설정
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

# 1. 자치구별 호텔 개수 집계
hotel_count = hotel[hotel['상세영업상태명'] == '영업중'].groupby('자치구').size().reset_index(name='호텔개수')

# 2. 방문자 데이터 요약 (자치구별 1년 치 총방문자수 합계 구하기)
visiter_summary = visiter_df.groupby('자치구')['총방문자수'].sum().reset_index()

# 3. 요약된 방문자 데이터와 호텔 개수 병합
hotel_merged_df = pd.merge(visiter_summary, hotel_count, on='자치구', how='left')

# 호텔이 하나도 없는 자치구는 결측치(NaN)가 되므로 0으로 채워줍니다.
hotel_merged_df['호텔개수'] = hotel_merged_df['호텔개수'].fillna(0)

# 4. 산점도 시각화
plt.figure(figsize=(12, 7))
sns.scatterplot(data=hotel_merged_df, x='호텔개수', y='총방문자수', s=100, color='royalblue', alpha=0.8)

# 점 옆에 자치구 이름 표시 (어떤 구인지 직관적으로 확인)
for idx, row in hotel_merged_df.iterrows():
    plt.text(row['호텔개수'] + 0.5, row['총방문자수'], row['자치구'], fontsize=9)

plt.title('서울시 자치구별 호텔 개수와 총 방문자 수의 관계')
plt.xlabel('영업 중인 호텔 개수 (개)')
plt.ylabel('연간 총 방문자 수 (명)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# 1. '도로명전체주소'에서 '자치구' 추출하여 새로운 컬럼 생성
# 문자열을 띄어쓰기 기준으로 나눈 뒤, 두 번째 위치(인덱스 1)에 있는 단어를 가져옵니다.
hotel['자치구'] = hotel['도로명전체주소'].str.split().str[1]

# 2. 자치구별 사업장 개수 확인 (데이터프레임 형태로 보기 좋게 정리)
# value_counts()를 사용하면 개수가 많은 순서대로(내림차순) 정렬되어 나옵니다.
district_counts = hotel['자치구'].value_counts().reset_index()

# 컬럼명 깔끔하게 변경
district_counts.columns = ['자치구', '사업장_개수']

# 결과 확인
district_counts

In [ ]:
SIGNGU_nm_to_id = gdf_SIGNGU.set_index("SIGNGU_NM")["SIGNGU_CD"].to_dict() # 행정동 이름(region_nm)을 키로, 행정동 코드(region_id)를 값으로 하는 매핑 딕셔너리 생성

district_counts["SIGNGU_CD"] = district_counts["자치구"].map(SIGNGU_nm_to_id) # 등록인구 데이터의 행정동 이름을 기준으로 행정동 코드(region_id)를 매칭하여 추가
district_counts.head()

In [ ]:
# 면적 데이터와 등록인구 데이터를 행정동 코드(region_id) 기준으로 병합 각 행정동에 해당하는 면적(km²)을 추가
district_counts = district_counts.merge(SIGNGU_area_df[["SIGNGU_CD", "area_km2"]], on="SIGNGU_CD", how="left")

# 인구밀도 계산 (1 km²당 인구 수) - 총인구 ÷ 면적 → 인구밀도 컬럼 생성
district_counts["base_density"] = district_counts["사업장_개수"] / district_counts["area_km2"]
district_counts.head()

In [ ]:
# 자치구 공간데이터(gdf_admin)와 인구밀도 테이블(pop_df)을 region_id 기준으로 병합
gdf_map = gdf_SIGNGU.merge(
    district_counts[["SIGNGU_CD", "base_density"]],
    on="SIGNGU_CD",
    how="left"
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

gdf_map.plot(
    column="base_density",      # 색칠 기준 컬럼 (인구밀도)
    cmap="OrRd",               # 색상 팔레트 (연한색 → 진한색)
    legend=True,               # 범례 표시
    edgecolor="black",
    linewidth=0.2,
    ax=ax
)

plt.title("서울시 자치구 관광숙박업소 밀도 단계구분도", fontsize=14)
plt.axis("off")
plt.show()

In [ ]:
district_counts.sort_values(by="base_density", ascending=False).reset_index(drop=True)

In [ ]:
district_counts.sort_values(by="사업장_개수", ascending=False).reset_index(drop=True)

In [ ]:
# 한글 폰트 설정 (깨짐 방지)
plt.rc('font', family='Malgun Gothic') # Mac은 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 배경 지도의 좌표계를 위도/경도(EPSG:4326)로 변환
gdf_map = gdf_map.to_crs(epsg=4326)

# 1. 그래프 도화지 크기 설정
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

# 2. [배경 레이어] 자치구별 밀도 단계구분도 그리기 (gdf_map)
# ⚠️ '밀도'라는 컬럼명은 실제 밀도값이 들어있는 컬럼명으로 바꿔주세요!
gdf_map.plot(column='base_density',
             ax=ax,
             cmap='OrRd', # 주황~빨강 그라데이션
             edgecolor='black', # 자치구 경계선 색상
             linewidth=0.5,
             legend=True,
             legend_kwds={'label': "관광숙박업소 밀도"})

# 3. [점 레이어] 활성 모텔 파란 점 찍기 (active_motels_clean)
# ⚠️ '경도', '위도' 컬럼명은 실제 좌표 컬럼명(예: lon, lat)으로 바꿔주세요!
ax.scatter(x=active_hotels_clean['X좌표(WGS84)'],  # 'Y좌표(WGS84)', 'X좌표(WGS84)'
           y=active_hotels_clean['Y좌표(WGS84)'],
           color='blue',
           s=15, # 점 크기
           alpha=0.6, # 투명도 (겹친 곳을 보기 위해 약간 투명하게)
           edgecolor='black',
           linewidth=0.5)

# 4. 지도 꾸미기
plt.title('서울시 자치구 관광숙박업소 밀도 및 활성 업소 분포도', fontsize=18, pad=20)
plt.axis('off') # 위경도 축 눈금 숨기기 (더 깔끔해짐)

# 5. 출력
plt.tight_layout()
plt.show()

In [ ]:
# 1. 서울 중심부(시청) 기준으로 기본 지도 띄우기
m = folium.Map(location=[37.5665, 126.9780], zoom_start=11)

# 2. [배경 레이어] 자치구 밀도 단계구분도 얹기
# ⚠️ '자치구', '밀도' 등 데이터에 맞는 정확한 컬럼명으로 수정해 주세요!
folium.Choropleth(
    geo_data=gdf_map,
    data=gdf_map,
    columns=['SIGNGU_NM', 'base_density'],
    key_on='feature.properties.SIGNGU_NM', # GeoJSON 구조에 따라 다를 수 있습니다
    fill_color='OrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='일반숙박업소 밀도'
).add_to(m)

# 3. [점 레이어] 활성 모텔 파란 점 얹기
# iterrows()를 돌면서 점을 하나씩 찍어줍니다.
for idx, row in active_hotels_clean.iterrows():
    folium.CircleMarker(
        location=[row['Y좌표(WGS84)'], row['X좌표(WGS84)']], # ⚠️ 실제 위도, 경도 컬럼명으로 수정
        radius=3, # 점 크기
        color='blue',
        fill=True,
        fill_color='blue',
        fill_opacity=0.6,
        popup=row.get('사업장명', '모텔') # 마커 클릭 시 이름 표시 (선택사항)
    ).add_to(m)

# 4. 지도 출력
m

## 행정동 경계와 결합

In [ ]:
# 정규표현식 r'\((.*?)\)'를 사용하여 괄호 안의 문자열만 추출합니다.
hotel['행정동'] = hotel['도로명전체주소'].str.extract(r'\((.*?)\)')

# 결과가 잘 나왔는지 확인 (관련 컬럼만 출력)
print(hotel[['도로명전체주소', '행정동']].head())

In [ ]:
hotel_df = hotel

hotel_df

In [ ]:
dong_nm_to_id = gdf_dong.set_index("ADSTRD_NM")["ADSTRD_CD"].to_dict() # 행정동 이름(region_nm)을 키로, 행정동 코드(region_id)를 값으로 하는 매핑 딕셔너리 생성

hotel_df["ADSTRD_CD"] = hotel_df["행정동"].map(dong_nm_to_id) # 등록인구 데이터의 행정동 이름을 기준으로 행정동 코드(region_id)를 매칭하여 추가
hotel_df.head()

In [ ]:
# 1. ADSTRD_CD 컬럼이 NaN인 전체 행(데이터)만 필터링해서 보기
nan_df = hotel_df[hotel_df['ADSTRD_CD'].isna()]
display(nan_df.head())

# 2. 매핑에 실패한 '행정동' 이름만 모아서 보기 (어떤 동들이 문제인지 파악)
# unique()를 사용해 중복 없이 실패한 동 이름만 리스트업 합니다.
missing_dongs = nan_df['행정동'].unique()
print(f"매핑 실패한 동 개수: {len(missing_dongs)}개")
print("매핑 실패한 동 목록:", missing_dongs)

# 3. (선택) 어떤 동이 가장 많이 누락되었는지 빈도수 확인
print("\n[누락된 동별 데이터 개수]")
print(nan_df['행정동'].value_counts())

In [ ]:
import geopandas as gp
from shapely import wkt

# ==========================================
# 🌟 에러 해결 핵심: 텍스트를 진짜 공간 객체로 변환
# ==========================================
# 기존 컬럼(텍스트)을 Shapely Geometry 객체로 덮어씌웁니다.
hotel_df['geometry'] = gpd.GeoSeries.from_wkt(hotel_df['geometry'])

# 이제 완벽한 GeoDataFrame으로 변환이 가능해집니다!
hotel_gdf = gpd.GeoDataFrame(hotel_df, geometry='geometry')

# ==========================================
# 2. 좌표계(CRS) 통일하기
# ==========================================
# 텍스트에서 방금 변환했기 때문에 hotel_gdf는 현재 좌표계 정보가 비어있을 확률이 높습니다.
if hotel_gdf.crs is None:
    # gdf_dong과 같은 좌표계(예: GRS80TM 등)라고 가정하고 강제로 이름을 붙여줍니다.
    hotel_gdf.set_crs(gdf_dong.crs, inplace=True)
elif hotel_gdf.crs != gdf_dong.crs:
    # 혹시 다르게 설정되어 있다면 gdf_dong 기준으로 변환해 줍니다.
    hotel_gdf = hotel_gdf.to_crs(gdf_dong.crs)

# ==========================================
# 3. 공간 조인 (Spatial Join) 실행
# ==========================================
# 기존 텍스트 매핑 컬럼이 있다면 삭제
if 'ADSTRD_CD' in hotel_gdf.columns:
    hotel_gdf = hotel_gdf.drop(columns=['ADSTRD_CD'])

hotel_final = gpd.sjoin(
    hotel_gdf,
    gdf_dong[['ADSTRD_CD', 'ADSTRD_NM', 'geometry']],
    how='left',
    predicate='within'
)

if 'index_right' in hotel_final.columns:
    hotel_final = hotel_final.drop(columns=['index_right'])

# 4. 결과 확인
print(f"✅ 원본 데이터 개수: {len(hotel_df)}개")
print(f"✅ 공간 조인 후 데이터 개수: {len(hotel_final)}개")
print(f"⚠️ 매핑 실패(결측치) 개수: {hotel_final['ADSTRD_CD'].isna().sum()}개")

display(hotel_final[['사업장명', '도로명전체주소', '행정동', 'ADSTRD_NM', 'ADSTRD_CD']].head(10))

In [ ]:
import folium
import geopandas as gpd

# ==========================================
# 1. Folium 지도용 좌표계(WGS84)로 변환
# ==========================================
hotel_map = hotel_final.to_crs(epsg=4326)
gdf_dong_map = gdf_dong.to_crs(epsg=4326)

# ==========================================
# 2. 행정동별 호텔 개수 집계 및 데이터 병합(Merge) 🌟
# ==========================================
dong_counts = hotel_map.groupby('ADSTRD_CD').size().reset_index(name='숙박업소_수')

# 툴팁에 띄우기 위해, 행정동 공간 데이터(gdf_dong_map)에 집계된 호텔 개수를 붙여줍니다.
gdf_dong_map = gdf_dong_map.merge(dong_counts, on='ADSTRD_CD', how='left')
# 호텔이 하나도 없는 동은 NaN이 되므로 0으로 채워줍니다.
gdf_dong_map['숙박업소_수'] = gdf_dong_map['숙박업소_수'].fillna(0).astype(int)

# ==========================================
# 3. 기본 지도 생성
# ==========================================
m = folium.Map(location=[37.5665, 126.9780], zoom_start=11)

# ==========================================
# 4. [레이어 1] 행정동별 밀도 코로플레스 (배경 색칠)
# ==========================================
folium.Choropleth(
    geo_data=gdf_dong_map,
    data=gdf_dong_map, # 병합된 데이터를 그대로 사용
    columns=['ADSTRD_CD', '숙박업소_수'],
    key_on='feature.properties.ADSTRD_CD',
    fill_color='YlOrRd',
    fill_opacity=0.6,
    line_opacity=0.3,
    legend_name='행정동별 관광숙박업소 밀도 (개수)'
).add_to(m)

# ==========================================
# 5. [레이어 2] 마우스 오버 말풍선(Tooltip) 추가 🌟
# ==========================================
# 투명한 GeoJson 레이어를 한 겹 덮어씌워서 마우스가 올라갈 때만 반응하게 만듭니다.
folium.GeoJson(
    gdf_dong_map,
    style_function=lambda x: {'fillColor': 'transparent', 'color': 'transparent'}, # 화면엔 안 보이게 투명 처리
    tooltip=folium.GeoJsonTooltip(
        fields=['ADSTRD_NM', '숙박업소_수'],   # 툴팁에 보여줄 데이터 컬럼
        aliases=['행정동 이름: ', '관광숙박업소 수: '], # 툴팁에 표시될 라벨(별명)
        style=("background-color: white; color: black; font-family: 'Malgun Gothic', sans-serif; font-size: 13px; font-weight: bold; padding: 10px; border-radius: 5px; box-shadow: 3px 3px 5px rgba(0,0,0,0.2);")
    )
).add_to(m)

# ==========================================
# 6. [레이어 3] 개별 관광숙박업소 위치 마커 (파란 점)
# ==========================================
valid_hotels = hotel_map.dropna(subset=['geometry'])
valid_hotels = valid_hotels[~valid_hotels.geometry.is_empty]

for idx, row in valid_hotels.iterrows():
    # geometry 객체에서 위도(y)와 경도(x) 값을 미리 뽑아냅니다.
    y = row['geometry'].y
    x = row['geometry'].x

    # 🌟 궁극의 방어막: y와 x값이 NaN(결측치)이 아닌 '정상적인 숫자'일 때만 마커를 찍습니다!
    if pd.notna(y) and pd.notna(x):
        folium.CircleMarker(
            location=[y, x],
            radius=2.5,
            color='darkblue',
            weight=0.5,
            fill=True,
            fill_color='blue',
            fill_opacity=0.8
        ).add_to(m)

# 7. 최종 지도 출력
m

In [ ]:
# 1. 지도 데이터에서 '행정동 이름'과 '숙박업소 수' 컬럼만 추출하여 일반 데이터프레임으로 변환
# (geometry 등 불필요한 공간 데이터는 제외하여 표를 깔끔하게 만듭니다)
rank_df = pd.DataFrame(gdf_dong_map[['ADSTRD_NM', '숙박업소_수']])

# ==========================================
# 2. 상위 TOP 10 (숙박업소가 가장 많은 동)
# ==========================================
# sort_values(ascending=False)로 내림차순 정렬 후 위에서 10개를 자릅니다.
top_10 = rank_df.sort_values(by='숙박업소_수', ascending=False).head(10).reset_index(drop=True)
top_10.index = range(1, 11) # 순위를 1위부터 10위까지 예쁘게 표시
top_10.columns = ['행정동명', '관광숙박업소 수 (개)']

print("🏆 [서울시 관광숙박업소 밀집 TOP 10 행정동]")
display(top_10)

# ==========================================
# 3. 하위 BOTTOM 10 (숙박업소가 가장 적은 동)
# ==========================================
# sort_values(ascending=True)로 오름차순 정렬 후 위에서 10개를 자릅니다.
bottom_10 = rank_df.sort_values(by=['숙박업소_수', 'ADSTRD_NM'], ascending=[True, True]).head(20).reset_index(drop=True)
bottom_10.index = range(1, 21)
bottom_10.columns = ['행정동명', '관광숙박업소 수 (개)']

print("\n📉 [서울시 관광숙박업소 사각지대 BOTTOM 20 행정동]")
display(bottom_10)

# 일반숙박업 인허가 정보

In [ ]:
BASE_DIR = os.getcwd()                     # 현재 작업 중인 기본 폴더 경로
FOLDER_DIR = os.path.join(BASE_DIR, "완전최종_전처리완료_CSV모음")  # 가공 데이터 저장 폴더

DATA_DIR = os.path.join(FOLDER_DIR, "서울시_일반숙박업_인허가_정보.csv")

motel = pd.read_csv(DATA_DIR)

In [ ]:
motel

In [ ]:
# '서울특별시 서초구 바우뫼로...' 에서 두 번째 단어(서초구) 추출
motel['자치구'] = motel['소재지전체주소'].dropna().apply(lambda x: x.split()[1] if len(x.split()) > 1 else None)

# 추출이 잘 되었는지 확인
print(motel['자치구'].unique())

In [ ]:
plt.rc('font', family='Malgun Gothic')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 숙박업 상세명 빈도 막대 그래프
sns.countplot(data=motel, y='숙박업상세명', order=motel['숙박업상세명'].value_counts().index, ax=axes[0], palette='pastel')
axes[0].set_title('숙박업 유형별 개수')

# 상세 영업 상태 막대 그래프
sns.countplot(data=motel, x='상세영업상태명', order=motel['상세영업상태명'].value_counts().index, ax=axes[1], palette='Set2')
axes[1].set_title('영업 상태별 개수')

plt.tight_layout()
plt.show()

In [ ]:
# 현재 '영업중'인 호텔만 필터링 (예시: 영업 상태명이 '영업'인 경우)
# 실제 데이터의 '영업' 상태명 텍스트에 맞게 수정하세요.
active_motels = motel[motel['상세영업상태명'] == '영업']

plt.figure(figsize=(10, 8))
# X좌표(경도), Y좌표(위도)를 이용해 산점도 시각화
sns.scatterplot(data=active_motels, x='X좌표(WGS84)', y='Y좌표(WGS84)',
                hue='숙박업상세명', alpha=0.6, s=50)

plt.title('서울시 영업 중인 일반숙박업소 위치 분포')
plt.xlabel('경도 (Longitude)')
plt.ylabel('위도 (Latitude)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
# 1. 서울 중심 좌표 설정 (위도: Y좌표, 경도: X좌표)
seoul_center = [37.5665, 126.9780]

# 2. Folium 지도 객체 생성 (zoom_start로 초기 확대 정도 설정)
m = folium.Map(location=seoul_center, zoom_start=11, tiles='CartoDB positron')
# tiles='CartoDB positron'은 지도를 깔끔한 밝은 톤으로 만들어주어 데이터(점)가 잘 보입니다.

# 3. 숙박업 유형별 마커 색상 지정 (Seaborn의 hue 역할)
# 데이터에 있는 유형들에 맞춰 원하는 색상으로 매핑합니다.
color_dict = {
    '여관업': 'blue',
    '관광호텔': 'orange',
    '여인숙업': 'green',
    '일반호텔': 'red',
    '숙박업(생활)': 'purple',
    '숙박업(기타)': 'brown',
    '휴양콘도미니엄업': 'pink'
}

# 4. 좌표 데이터가 없는 결측치(NaN) 행 제거 (에러 방지)
active_motels_clean = active_motels.dropna(subset=['Y좌표(WGS84)', 'X좌표(WGS84)'])

# 5. 데이터를 반복하며 지도에 점(CircleMarker) 찍기
for idx, row in active_motels_clean.iterrows():
    hotel_type = row['숙박업상세명']

    # 딕셔너리에서 색상을 찾고, 매핑되지 않은 유형은 'gray'로 처리
    marker_color = color_dict.get(hotel_type, 'gray')

    # 마커 클릭 시 보여줄 정보 (사업장명과 숙박업 유형)
    popup_text = f"<b>{row['사업장명']}</b><br>{hotel_type}"

    folium.CircleMarker(
        # folium은 반드시 [위도(Y), 경도(X)] 순서로 입력해야 합니다!
        location=[row['Y좌표(WGS84)'], row['X좌표(WGS84)']],
        radius=4,               # 점의 크기
        color=marker_color,     # 테두리 색상
        fill=True,              # 안쪽 채우기
        fill_color=marker_color,# 안쪽 색상
        fill_opacity=0.7,       # 투명도
        popup=folium.Popup(popup_text, max_width=250) # 클릭 시 팝업창
    ).add_to(m)

# 6. 지도 출력 (주피터 노트북 환경에서는 m을 입력하면 지도가 바로 뜹니다)
m

In [ ]:
# 1. 서울 중심 좌표 설정 (이전과 동일)
seoul_center = [37.5665, 126.9780]

# 2. Folium 지도 객체 생성 (이전과 동일한 지도 스타일)
m = folium.Map(location=seoul_center, zoom_start=11)
# 참고: 마커가 훨씬 더 잘 보이게 하려면 지도 배경을 단순한 흰색 계열로 바꾸는 것도 좋습니다.
# m = folium.Map(location=seoul_center, zoom_start=11, tiles='CartoDB positron')

# 3. 영업 중인 호텔만 지도에 마커 추가
# 좌표 데이터가 없는 결측치(NaN) 행 제거
active_motels_clean = active_motels.dropna(subset=['Y좌표(WGS84)', 'X좌표(WGS84)'])

for idx, row in active_motels_clean.iterrows():
    folium.CircleMarker(
        # [위도, 경도] 순서 중요!
        location=[row['Y좌표(WGS84)'], row['X좌표(WGS84)']],

        # --- 여기부터 마커 스타일 수정 ---
        radius=5,              # 마커 크기 확대 (기존 3 -> 5)
        color='darkblue',      # 테두리 색상을 더 진한 파란색으로 변경
        weight=2,              # 테두리 선 두께 증가
        fill=True,
        fill_color='dodgerblue', # 내부 색상을 더 밝고 선명한 파란색으로
        fill_opacity=0.8,      # 투명도 조절 (기존 0.6 -> 0.8, 덜 투명하게)
        # --------------------------------

        popup=row['사업장명']
    ).add_to(m)

# 4. 지도 출력 (주피터 노트북 환경)
m

In [ ]:
# 한글 폰트 및 마이너스 깨짐 방지 설정
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

# 1. 자치구별 호텔 개수 집계
motel_count = motel[motel['상세영업상태명'] == '영업'].groupby('자치구').size().reset_index(name='모텔개수')

# 2. 방문자 데이터 요약 (자치구별 1년 치 총방문자수 합계 구하기)
visiter_summary = visiter_df.groupby('자치구')['총방문자수'].sum().reset_index()

# 3. 요약된 방문자 데이터와 호텔 개수 병합
motel_merged_df = pd.merge(visiter_summary, motel_count, on='자치구', how='left')

# 호텔이 하나도 없는 자치구는 결측치(NaN)가 되므로 0으로 채워줍니다.
motel_merged_df['모텔개수'] = motel_merged_df['모텔개수'].fillna(0)

# 4. 산점도 시각화
plt.figure(figsize=(12, 7))
sns.scatterplot(data=motel_merged_df, x='모텔개수', y='총방문자수', s=100, color='royalblue', alpha=0.8)

# 점 옆에 자치구 이름 표시 (어떤 구인지 직관적으로 확인)
for idx, row in motel_merged_df.iterrows():
    plt.text(row['모텔개수'] + 0.5, row['총방문자수'], row['자치구'], fontsize=9)

plt.title('서울시 자치구별 모텔 개수와 총 방문자 수의 관계')
plt.xlabel('영업 중인 모텔 개수 (개)')
plt.ylabel('연간 총 방문자 수 (명)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# 1. '도로명전체주소'에서 '자치구' 추출하여 새로운 컬럼 생성
# 문자열을 띄어쓰기 기준으로 나눈 뒤, 두 번째 위치(인덱스 1)에 있는 단어를 가져옵니다.
motel['자치구'] = motel['도로명전체주소'].str.split().str[1]

# 2. 자치구별 사업장 개수 확인 (데이터프레임 형태로 보기 좋게 정리)
# value_counts()를 사용하면 개수가 많은 순서대로(내림차순) 정렬되어 나옵니다.
motel_counts = motel['자치구'].value_counts().reset_index()

# 컬럼명 깔끔하게 변경
motel_counts.columns = ['자치구', '사업장_개수']

# 결과 확인
motel_counts

In [ ]:
SIGNGU_nm_to_id = gdf_SIGNGU.set_index("SIGNGU_NM")["SIGNGU_CD"].to_dict() # 행정동 이름(region_nm)을 키로, 행정동 코드(region_id)를 값으로 하는 매핑 딕셔너리 생성

motel_counts["SIGNGU_CD"] = motel_counts["자치구"].map(SIGNGU_nm_to_id) # 등록인구 데이터의 행정동 이름을 기준으로 행정동 코드(region_id)를 매칭하여 추가
motel_counts.head()

In [ ]:
# 면적 데이터와 등록인구 데이터를 행정동 코드(region_id) 기준으로 병합 각 행정동에 해당하는 면적(km²)을 추가
motel_counts = motel_counts.merge(SIGNGU_area_df[["SIGNGU_CD", "area_km2"]], on="SIGNGU_CD", how="left")

# 인구밀도 계산 (1 km²당 인구 수) - 총인구 ÷ 면적 → 인구밀도 컬럼 생성
motel_counts["base_density"] = motel_counts["사업장_개수"] / district_counts["area_km2"]
motel_counts.head()

In [ ]:
# 자치구 공간데이터(gdf_admin)와 인구밀도 테이블(pop_df)을 region_id 기준으로 병합
gdf_map = gdf_SIGNGU.merge(
    motel_counts[["SIGNGU_CD", "base_density"]],
    on="SIGNGU_CD",
    how="left"
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

gdf_map.plot(
    column="base_density",      # 색칠 기준 컬럼 (인구밀도)
    cmap="OrRd",               # 색상 팔레트 (연한색 → 진한색)
    legend=True,               # 범례 표시
    edgecolor="black",
    linewidth=0.2,
    ax=ax
)

plt.title("서울시 자치구 일반숙박업소 밀도 단계구분도", fontsize=14)
plt.axis("off")
plt.show()

In [ ]:
motel_counts.sort_values(by="base_density", ascending=False).reset_index(drop=True)

In [ ]:
motel_counts.sort_values(by="사업장_개수", ascending=False).reset_index(drop=True)

In [ ]:
# 한글 폰트 설정 (깨짐 방지)
plt.rc('font', family='Malgun Gothic') # Mac은 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 배경 지도의 좌표계를 위도/경도(EPSG:4326)로 변환
gdf_map = gdf_map.to_crs(epsg=4326)

# 1. 그래프 도화지 크기 설정
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

# 2. [배경 레이어] 자치구별 밀도 단계구분도 그리기 (gdf_map)
# ⚠️ '밀도'라는 컬럼명은 실제 밀도값이 들어있는 컬럼명으로 바꿔주세요!
gdf_map.plot(column='base_density',
             ax=ax,
             cmap='OrRd', # 주황~빨강 그라데이션
             edgecolor='black', # 자치구 경계선 색상
             linewidth=0.5,
             legend=True,
             legend_kwds={'label': "일반숙박업소 밀도"})

# 3. [점 레이어] 활성 모텔 파란 점 찍기 (active_motels_clean)
# ⚠️ '경도', '위도' 컬럼명은 실제 좌표 컬럼명(예: lon, lat)으로 바꿔주세요!
ax.scatter(x=active_motels_clean['X좌표(WGS84)'],  # 'Y좌표(WGS84)', 'X좌표(WGS84)'
           y=active_motels_clean['Y좌표(WGS84)'],
           color='blue',
           s=15, # 점 크기
           alpha=0.6, # 투명도 (겹친 곳을 보기 위해 약간 투명하게)
           edgecolor='black',
           linewidth=0.5)

# 4. 지도 꾸미기
plt.title('서울시 자치구 일반숙박업소 밀도 및 활성 업소 분포도', fontsize=18, pad=20)
plt.axis('off') # 위경도 축 눈금 숨기기 (더 깔끔해짐)

# 5. 출력
plt.tight_layout()
plt.show()

In [ ]:
# 1. 서울 중심부(시청) 기준으로 기본 지도 띄우기
m = folium.Map(location=[37.5665, 126.9780], zoom_start=11)

# 2. [배경 레이어] 자치구 밀도 단계구분도 얹기
# ⚠️ '자치구', '밀도' 등 데이터에 맞는 정확한 컬럼명으로 수정해 주세요!
folium.Choropleth(
    geo_data=gdf_map,
    data=gdf_map,
    columns=['SIGNGU_NM', 'base_density'],
    key_on='feature.properties.SIGNGU_NM', # GeoJSON 구조에 따라 다를 수 있습니다
    fill_color='OrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='일반숙박업소 밀도'
).add_to(m)

# 3. [점 레이어] 활성 모텔 파란 점 얹기
# iterrows()를 돌면서 점을 하나씩 찍어줍니다.
for idx, row in active_motels_clean.iterrows():
    folium.CircleMarker(
        location=[row['Y좌표(WGS84)'], row['X좌표(WGS84)']], # ⚠️ 실제 위도, 경도 컬럼명으로 수정
        radius=3, # 점 크기
        color='blue',
        fill=True,
        fill_color='blue',
        fill_opacity=0.6,
        popup=row.get('사업장명', '모텔') # 마커 클릭 시 이름 표시 (선택사항)
    ).add_to(m)

# 4. 지도 출력
m

## 행정동 경계와 결합

In [ ]:
# 정규표현식 r'\((.*?)\)'를 사용하여 괄호 안의 문자열만 추출합니다.
motel['행정동'] = motel['도로명전체주소'].str.extract(r'\((.*?)\)')

# 결과가 잘 나왔는지 확인 (관련 컬럼만 출력)
print(motel[['도로명전체주소', '행정동']].head())

In [ ]:
motel_df = motel

motel_df

In [ ]:
# ==========================================
# 🌟 에러 해결 핵심: 텍스트를 진짜 공간 객체로 변환
# ==========================================
# 기존 컬럼(텍스트)을 Shapely Geometry 객체로 덮어씌웁니다.
motel_df['geometry'] = gpd.GeoSeries.from_wkt(motel_df['geometry'])

# 이제 완벽한 GeoDataFrame으로 변환이 가능해집니다!
motel_gdf = gpd.GeoDataFrame(motel_df, geometry='geometry')

# ==========================================
# 2. 좌표계(CRS) 통일하기
# ==========================================
# 텍스트에서 방금 변환했기 때문에 hotel_gdf는 현재 좌표계 정보가 비어있을 확률이 높습니다.
if motel_gdf.crs is None:
    # gdf_dong과 같은 좌표계(예: GRS80TM 등)라고 가정하고 강제로 이름을 붙여줍니다.
    motel_gdf.set_crs(gdf_dong.crs, inplace=True)
elif motel_gdf.crs != gdf_dong.crs:
    # 혹시 다르게 설정되어 있다면 gdf_dong 기준으로 변환해 줍니다.
    motel_gdf = motel_gdf.to_crs(gdf_dong.crs)

# ==========================================
# 3. 공간 조인 (Spatial Join) 실행
# ==========================================
# 기존 텍스트 매핑 컬럼이 있다면 삭제
if 'ADSTRD_CD' in motel_gdf.columns:
    motel_gdf = motel_gdf.drop(columns=['ADSTRD_CD'])

motel_final = gpd.sjoin(
    motel_gdf,
    gdf_dong[['ADSTRD_CD', 'ADSTRD_NM', 'geometry']],
    how='left',
    predicate='within'
)

if 'index_right' in motel_final.columns:
    motel_final = motel_final.drop(columns=['index_right'])

# 4. 결과 확인
print(f"✅ 원본 데이터 개수: {len(motel_df)}개")
print(f"✅ 공간 조인 후 데이터 개수: {len(motel_final)}개")
print(f"⚠️ 매핑 실패(결측치) 개수: {motel_final['ADSTRD_CD'].isna().sum()}개")

display(motel_final[['사업장명', '도로명전체주소', '행정동', 'ADSTRD_NM', 'ADSTRD_CD']].head(10))

In [ ]:
import folium
import geopandas as gpd

# ==========================================
# 1. Folium 지도용 좌표계(WGS84)로 변환
# ==========================================
motel_map = motel_final.to_crs(epsg=4326)
gdf_dong_map = gdf_dong.to_crs(epsg=4326)

# ==========================================
# 2. 행정동별 호텔 개수 집계 및 데이터 병합(Merge) 🌟
# ==========================================
dong_counts = motel_map.groupby('ADSTRD_CD').size().reset_index(name='숙박업소_수')

# 툴팁에 띄우기 위해, 행정동 공간 데이터(gdf_dong_map)에 집계된 호텔 개수를 붙여줍니다.
gdf_dong_map = gdf_dong_map.merge(dong_counts, on='ADSTRD_CD', how='left')
# 호텔이 하나도 없는 동은 NaN이 되므로 0으로 채워줍니다.
gdf_dong_map['숙박업소_수'] = gdf_dong_map['숙박업소_수'].fillna(0).astype(int)

# ==========================================
# 3. 기본 지도 생성
# ==========================================
m = folium.Map(location=[37.5665, 126.9780], zoom_start=11)

# ==========================================
# 4. [레이어 1] 행정동별 밀도 코로플레스 (배경 색칠)
# ==========================================
folium.Choropleth(
    geo_data=gdf_dong_map,
    data=gdf_dong_map, # 병합된 데이터를 그대로 사용
    columns=['ADSTRD_CD', '숙박업소_수'],
    key_on='feature.properties.ADSTRD_CD',
    fill_color='YlOrRd',
    fill_opacity=0.6,
    line_opacity=0.3,
    legend_name='행정동별 일반숙박업소 밀도 (개수)'
).add_to(m)

# ==========================================
# 5. [레이어 2] 마우스 오버 말풍선(Tooltip) 추가 🌟
# ==========================================
# 투명한 GeoJson 레이어를 한 겹 덮어씌워서 마우스가 올라갈 때만 반응하게 만듭니다.
folium.GeoJson(
    gdf_dong_map,
    style_function=lambda x: {'fillColor': 'transparent', 'color': 'transparent'}, # 화면엔 안 보이게 투명 처리
    tooltip=folium.GeoJsonTooltip(
        fields=['ADSTRD_NM', '숙박업소_수'],   # 툴팁에 보여줄 데이터 컬럼
        aliases=['행정동 이름: ', '일반숙박업소 수: '], # 툴팁에 표시될 라벨(별명)
        style=("background-color: white; color: black; font-family: 'Malgun Gothic', sans-serif; font-size: 13px; font-weight: bold; padding: 10px; border-radius: 5px; box-shadow: 3px 3px 5px rgba(0,0,0,0.2);")
    )
).add_to(m)

# ==========================================
# 6. [레이어 3] 개별 관광숙박업소 위치 마커 (파란 점)
# ==========================================
valid_motels = motel_map.dropna(subset=['geometry'])
valid_motels = valid_motels[~valid_motels.geometry.is_empty]

for idx, row in valid_motels.iterrows():
    # geometry 객체에서 위도(y)와 경도(x) 값을 미리 뽑아냅니다.
    y = row['geometry'].y
    x = row['geometry'].x

    # 🌟 궁극의 방어막: y와 x값이 NaN(결측치)이 아닌 '정상적인 숫자'일 때만 마커를 찍습니다!
    if pd.notna(y) and pd.notna(x):
        folium.CircleMarker(
            location=[y, x],
            radius=2.5,
            color='darkblue',
            weight=0.5,
            fill=True,
            fill_color='blue',
            fill_opacity=0.8
        ).add_to(m)

# 7. 최종 지도 출력
m

In [ ]:
# 1. 지도 데이터에서 '행정동 이름'과 '숙박업소 수' 컬럼만 추출하여 일반 데이터프레임으로 변환
# (geometry 등 불필요한 공간 데이터는 제외하여 표를 깔끔하게 만듭니다)
rank_df = pd.DataFrame(gdf_dong_map[['ADSTRD_NM', '숙박업소_수']])

# ==========================================
# 2. 상위 TOP 10 (숙박업소가 가장 많은 동)
# ==========================================
# sort_values(ascending=False)로 내림차순 정렬 후 위에서 10개를 자릅니다.
top_10 = rank_df.sort_values(by='숙박업소_수', ascending=False).head(10).reset_index(drop=True)
top_10.index = range(1, 11) # 순위를 1위부터 10위까지 예쁘게 표시
top_10.columns = ['행정동명', '일반숙박업소 수 (개)']

print("🏆 [서울시 일반숙박업소 밀집 TOP 10 행정동]")
display(top_10)

# ==========================================
# 3. 하위 BOTTOM 10 (숙박업소가 가장 적은 동)
# ==========================================
# sort_values(ascending=True)로 오름차순 정렬 후 위에서 10개를 자릅니다.
bottom_10 = rank_df.sort_values(by=['숙박업소_수', 'ADSTRD_NM'], ascending=[True, True]).head(20).reset_index(drop=True)
bottom_10.index = range(1, 21)
bottom_10.columns = ['행정동명', '일반숙박업소 수 (개)']

print("\n📉 [서울시 일반숙박업소 사각지대 BOTTOM 20 행정동]")
display(bottom_10)

# 키워드별 언급량

In [ ]:
BASE_DIR = os.getcwd()                     # 현재 작업 중인 기본 폴더 경로
FOLDER_DIR = os.path.join(BASE_DIR, "완전최종_전처리완료_CSV모음")  # 가공 데이터 저장 폴더

DATA_DIR = os.path.join(FOLDER_DIR, "키워드별_언급량.csv")

keyword = pd.read_csv(DATA_DIR)

In [ ]:
keyword

In [ ]:
# 한글 폰트 및 마이너스 깨짐 방지 설정
plt.rc('font', family='Malgun Gothic') # Mac은 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 1. 키워드별 전체 언급량 합계 계산 및 내림차순 정렬
total_keywords = keyword.groupby('키워드')['언급량'].sum().sort_values(ascending=False).reset_index()

# 2. 막대 그래프 시각화
plt.figure(figsize=(12, 6))
sns.barplot(data=total_keywords, x='언급량', y='키워드', palette='viridis')
plt.title('2025년 K-컬처 키워드별 누적 언급량 순위')
plt.xlabel('누적 언급량')
plt.ylabel('키워드')
plt.show()

In [ ]:
total_keywords

In [ ]:
plt.figure(figsize=(14, 7))
# 월별 변화를 선 그래프로 시각화 (hue를 키워드로 지정하여 색상 구분)
sns.lineplot(data=keyword, x='월', y='언급량', hue='키워드', marker='o', linewidth=2)

plt.title('2025년 월별 K-컬처 키워드 언급량 추이')
plt.xlabel('월')
plt.ylabel('언급량')
plt.xticks(range(1, 13)) # x축을 1~12월로 고정
# 범례가 그래프를 가리지 않도록 바깥으로 빼주기
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# 1. 월을 열(Column)로, 키워드를 행(Row)으로 하는 피벗 테이블 생성 (값은 '월별순위')
rank_pivot = keyword.pivot(index='키워드', columns='월', values='월별순위')

# 2. 히트맵 시각화
plt.figure(figsize=(12, 8))
# 순위는 숫자가 작을수록(1위) 높은 것이므로 _r이 붙은 색상표(역순)를 사용합니다.
sns.heatmap(rank_pivot, annot=True, fmt=".0f", cmap='YlGnBu_r', linewidths=0.5, cbar_kws={'label': '순위'})

plt.title('2025년 K-컬처 키워드 월별 순위 변동')
plt.xlabel('월')
plt.ylabel('키워드')
plt.show()

In [ ]:
# 1. 한글 폰트 설정 (깨짐 방지용)
# 운영체제에 맞게 주석을 해제해서 사용하세요.
plt.rc('font', family='Malgun Gothic') # Windows

plt.rcParams['axes.unicode_minus'] = False

# 2. X축 예쁘게 출력을 위해 '기준년월'을 문자열(String)로 변환
keyword['기준년월'] = keyword['기준년월'].astype(str)

# 3. 그래프 사이즈 설정
plt.figure(figsize=(14, 7))

# 4. Seaborn으로 라인 그래프 그리기
# 마커(점)를 표시하고, 선 스타일을 통일하기 위해 dashes=False 설정
sns.lineplot(data=keyword,
             x='기준년월',
             y='월별순위',
             hue='키워드',
             style='키워드',
             markers=True,
             dashes=False,
             palette='tab10', # 색상 테마 설정
             linewidth=2,
             markersize=10)

# 5. 핵심 포인트! Y축 뒤집기 (1위가 맨 위로 오도록)
plt.gca().invert_yaxis()

# 6. Y축 눈금(Tick)을 1위부터 10위까지 정수로 설정
plt.yticks(range(1, 11), [f'{i}위' for i in range(1, 11)])

# 7. 제목 및 축 라벨 설정
plt.title('키워드별 순위 변동 추이', fontsize=18, fontweight='bold', loc='left', pad=20)
plt.xlabel('') # X축 라벨 숨김 (기준년월이 직관적이므로)
plt.ylabel('') # Y축 라벨 숨김

# 8. X축 글자 살짝 기울이기 (글자가 겹치지 않게)
plt.xticks(rotation=45)

# 9. 배경 그리드(격자) 추가
plt.grid(True, axis='both', linestyle='--', alpha=0.5)

# 10. 범례(Legend)를 그래프 오른쪽 밖으로 빼기
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., title='키워드')

# 그래프 여백 깔끔하게 자동 조정 후 출력
plt.tight_layout()
plt.show()

# 언급량 인게이지먼트

## 언급량 인게이지먼트 추이

In [ ]:
BASE_DIR = os.getcwd()                     # 현재 작업 중인 기본 폴더 경로
FOLDER_DIR = os.path.join(BASE_DIR, "완전최종_전처리완료_CSV모음")  # 가공 데이터 저장 폴더

DATA_DIR = os.path.join(FOLDER_DIR, "한국관광관련_언급량_인게이지먼트_추이.csv")

mention = pd.read_csv(DATA_DIR)

mention

In [ ]:
# 1. 한글 폰트 설정
plt.rc('font', family='Malgun Gothic') # Windows
# plt.rc('font', family='AppleGothic') # Mac
plt.rcParams['axes.unicode_minus'] = False

# 2. X축 기준년월 문자열 생성
# mention['기준년월_str'] = mention['기준년월'].astype(str).str[2:4] + "년 " + mention['기준년월'].astype(str).str[4:] + "월"

# 🌟 핵심 해결책: X축을 문자열 대신 0~11의 숫자로 확실히 고정합니다.
x_pos = range(len(mention))

# 3. 도화지 세팅
fig, ax1 = plt.subplots(figsize=(14, 8))

# ==========================================
# [왼쪽 Y축] 언급량 그래프 그리기 (X축에 x_pos 사용)
# ==========================================
color1 = 'mediumseagreen'
ax1.plot(x_pos, mention['언급량'],
         marker='s', color=color1, linewidth=2.5, markersize=8, label='언급량')
ax1.set_xlabel('기준년월', fontsize=12)
ax1.set_ylabel('언급량 (건)', color=color1, fontsize=12, fontweight='bold')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.grid(True, linestyle='--', alpha=0.5)

def format_mentions(x, pos):
    return f'{int(x/10000)}만'
ax1.yaxis.set_major_formatter(FuncFormatter(format_mentions))

# ==========================================
# [오른쪽 Y축] 인게이지먼트 그래프 그리기 (X축에 x_pos 사용)
# ==========================================
ax2 = ax1.twinx()
color2 = 'darkorange'
ax2.plot(x_pos, mention['인게이지먼트'],
         marker='^', color=color2, linewidth=2.5, markersize=8, label='인게이지먼트')
ax2.set_ylabel('인게이지먼트 (건)', color=color2, fontsize=12, fontweight='bold')
ax2.tick_params(axis='y', labelcolor=color2)

def format_engagements(x, pos):
    return f'{x/10000:,.0f}만'
ax2.yaxis.set_major_formatter(FuncFormatter(format_engagements))

# ==========================================
# 🌟 안정적인 여백 및 X축 라벨 씌우기
# ==========================================
# 이제 x축이 완벽한 숫자이므로 여백(xlim)이 안전하게 먹힙니다.
ax1.set_xlim(-0.5, len(mention) - 0.5)
ax1.set_ylim(mention['언급량'].min() * 0.85, mention['언급량'].max() * 1.15)
ax2.set_ylim(mention['인게이지먼트'].min() * 0.85, mention['인게이지먼트'].max() * 1.15)

# 숫자였던 X축 눈금에 '25년 01월' 등의 예쁜 글씨 라벨을 덮어씌웁니다.
ax1.set_xticks(x_pos)
ax1.set_xticklabels(mention['월'])

# ==========================================
# 수치 레이블 달기 (고정 픽셀)
# ==========================================
for i in range(len(mention)):
    val1 = mention['언급량'].iloc[i]
    ax1.annotate(f"{val1/10000:.0f}만",
                 xy=(i, val1),
                 xytext=(0, 12), textcoords='offset points',
                 ha='center', va='bottom', fontsize=10, color=color1, fontweight='bold')

    val2 = mention['인게이지먼트'].iloc[i]
    ax2.annotate(f"{val2/10000:,.0f}만",
                 xy=(i, val2),
                 xytext=(0, -12), textcoords='offset points',
                 ha='center', va='top', fontsize=10, color=color2, fontweight='bold')

# ==========================================
# 4. 범례 및 타이틀 설정
# ==========================================
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left', bbox_to_anchor=(0.01, 0.99), fontsize=11)

plt.title('2025년 월별 언급량 및 인게이지먼트 추이 (절대치)', fontsize=18, fontweight='bold', pad=25)

plt.tight_layout()
plt.show()

In [ ]:
# 🌟 핵심 라이브러리 임포트!
# from adjustText import adjust_text

# 불필요한 경고 메시지 숨기기
warnings.filterwarnings('ignore')

# 1. 한글 폰트 설정
plt.rc('font', family='Malgun Gothic') # Windows
# plt.rc('font', family='AppleGothic') # Mac
plt.rcParams['axes.unicode_minus'] = False

# 2. X축 기준년월 문자열 생성
mention['기준년월_str'] = mention['기준년월'].astype(str).str[2:4] + "년 " + mention['기준년월'].astype(str).str[4:] + "월"

# 🌟 핵심 해결책: X축을 문자열 대신 0~11의 숫자로 확실히 고정합니다.
x_pos = range(len(mention))

# 3. 도화지 세팅
fig, axes = plt.subplots(2, 1, figsize=(14, 14))

# ==========================================
# [그래프 1] 긍정 vs 부정 여론 비율 추이
# ==========================================
# 🌟 x_pos 숫자 좌표로 그래프 그리기
axes[0].plot(x_pos, mention['긍정언급량 비율'], marker='o', color='royalblue', linewidth=2.5, markersize=8, label='긍정 비율(%)')
axes[0].plot(x_pos, mention['부정언급량 비율'], marker='o', color='tomato', linewidth=2.5, markersize=8, label='부정 비율(%)')

for i in range(len(mention)):
    # i는 숫자 인덱스이므로 그대로 사용 가능
    axes[0].text(i, mention['긍정언급량 비율'].iloc[i] + 1.5, f"{mention['긍정언급량 비율'].iloc[i]:.1f}%", ha='center', va='bottom', fontsize=10, color='royalblue', fontweight='bold')
    axes[0].text(i, mention['부정언급량 비율'].iloc[i] - 1.5, f"{mention['부정언급량 비율'].iloc[i]:.1f}%", ha='center', va='top', fontsize=10, color='tomato', fontweight='bold')

axes[0].set_title('2025년 월별 글로벌 긍정 vs 부정 여론 추이', fontsize=18, fontweight='bold', pad=15)
axes[0].set_ylabel('비율 (%)', fontsize=12)
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend(fontsize=12, loc='center right', bbox_to_anchor=(1.12, 0.5))

# 🌟 여백 확보 및 위쪽 그래프 X축 라벨 숨기기 (깔끔함을 위해)
axes[0].set_xlim(-0.5, len(mention) - 0.5)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([]) # 위쪽 그래프는 X축 글씨 숨김

# ==========================================
# [그래프 2] 언급량 및 인게이지먼트 증감률 추이
# ==========================================
# 🌟 x_pos 숫자 좌표로 그래프 그리기
axes[1].plot(x_pos, mention['언급량 증감률'], marker='s', color='mediumseagreen', linewidth=2.5, markersize=8, label='언급량 증감률(%)')
axes[1].plot(x_pos, mention['인게이지먼트 증감률'], marker='^', color='orange', linewidth=2.5, markersize=8, label='인게이지먼트 증감률(%)')
axes[1].axhline(0, color='black', linewidth=1.5, linestyle='--')

texts = []
for i in range(len(mention)):
    # i는 숫자 인덱스이므로 그대로 사용 가능
    t1 = axes[1].text(i, mention['언급량 증감률'].iloc[i],
                      f"{mention['언급량 증감률'].iloc[i]:.1f}%",
                      ha='center', fontsize=10, color='seagreen', fontweight='bold')
    texts.append(t1)

    t2 = axes[1].text(i, mention['인게이지먼트 증감률'].iloc[i],
                      f"{mention['인게이지먼트 증감률'].iloc[i]:.1f}%",
                      ha='center', fontsize=10, color='darkorange', fontweight='bold')
    texts.append(t2)

# adjust_text 실행 (텍스트 겹침 방지)
adjust_text(texts, ax=axes[1],
            arrowprops=dict(arrowstyle='-', color='grey', lw=0.8),
            expand_points=(1.5, 1.5))

axes[1].set_title('2025년 월별 언급량 및 인게이지먼트 증감률 추이', fontsize=18, fontweight='bold', pad=15)
axes[1].set_ylabel('증감률 (%)', fontsize=12)
axes[1].set_xlabel('기준년월', fontsize=12)
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(fontsize=12, loc='center right', bbox_to_anchor=(1.15, 0.5))

# 🌟 여백 확보 및 아래쪽 그래프 X축 라벨 덮어쓰기!
axes[1].set_xlim(-0.5, len(mention) - 0.5)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(mention['월']) # '25년 1월' 등 글씨 적용

plt.tight_layout()
plt.show()

## 언급량 인게이지먼트 잠재적 노출량

In [ ]:
BASE_DIR = os.getcwd()                     # 현재 작업 중인 기본 폴더 경로
FOLDER_DIR = os.path.join(BASE_DIR, "완전최종_전처리완료_CSV모음")  # 가공 데이터 저장 폴더

DATA_DIR = os.path.join(FOLDER_DIR, "한국관광관련_국가별_언급량_인게이지먼트_잠재적_노출량.csv")

mention2 = pd.read_csv(DATA_DIR)

mention2

In [ ]:
mention2.sort_values(by="잠재적 노출량", ascending=False).reset_index(drop=True)

In [ ]:
# 1. 한글 폰트 설정 (깨짐 방지용)
# 운영체제에 맞게 주석을 해제해서 사용하세요.
plt.rc('font', family='Malgun Gothic') # Windows
# plt.rc('font', family='AppleGothic') # Mac
plt.rcParams['axes.unicode_minus'] = False

# 2. 파생 변수 생성: '언급당 인게이지먼트' (글 1개를 쓸 때 얼마나 많은 호응을 이끌어내는가?)
mention2['언급당_인게이지먼트'] = mention2['인게이지먼트'] / mention2['언급량']

# 3. 도화지(Figure) 세팅
plt.figure(figsize=(14, 10))

# 4. 버블 크기 스케일링
# '잠재적 노출량' 단위가 수억~수십억 단위로 매우 크기 때문에 화면에 맞게 나눠줍니다.
# (버블이 너무 크거나 작으면 이 20000000 숫자를 조절해 보세요!)
bubble_sizes = mention2['잠재적 노출량'] / 2000000

# 5. 산점도(버블 차트) 그리기
scatter = plt.scatter(x=mention2['언급량'],
                      y=mention2['인게이지먼트'],
                      s=bubble_sizes,
                      c=mention2['언급당_인게이지먼트'], # 색상: 붉은색일수록 진성 팬(효율 최고!)
                      cmap='Reds', # 하얀색 -> 붉은색 그라데이션
                      alpha=0.7,
                      edgecolors='grey',
                      linewidth=1.2)

# 6. 버블 중앙에 국가명 텍스트 라벨 달기
for i in range(len(mention2)):
    plt.text(mention2['언급량'].iloc[i],
             mention2['인게이지먼트'].iloc[i],
             mention2['국가'].iloc[i],
             fontsize=11,
             ha='center',
             va='center',
             fontweight='bold')

# 7. 4사분면을 나누는 평균 십자선 긋기
plt.axvline(mention2['언급량'].mean(), color='black', linestyle='--', alpha=0.5)
plt.axhline(mention2['인게이지먼트'].mean(), color='black', linestyle='--', alpha=0.5)

# 8. 그래프 꾸미기
plt.title('국가별 K-컬처 파급력 및 진성 팬덤 분석', fontsize=20, fontweight='bold', pad=20)
plt.xlabel('언급량 (Volume)', fontsize=14)
plt.ylabel('인게이지먼트 (Engagement)', fontsize=14)

# 컬러바(범례) 추가
cbar = plt.colorbar(scatter)
cbar.set_label('언급당 인게이지먼트 (붉을수록 효율 높음)', fontsize=12)

plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# 1. 한글 폰트 설정 (운영체제에 맞게 주석 해제)
plt.rc('font', family='Malgun Gothic') # Windows용
# plt.rc('font', family='AppleGothic') # Mac용
plt.rcParams['axes.unicode_minus'] = False

# 2. 도화지 크기 설정
plt.figure(figsize=(12, 7))

# 3. 버블 크기 조절용 스케일 팩터
scale_factor = 1500

# 4. 각 국가별로 점(버블) 찍기 및 텍스트 추가
for idx, row in mention2.iterrows():
    # 🔴 버블 그리기
    plt.scatter(x=row['인게이지먼트'],
                y=row['잠재적 노출량'],
                s=row['언급량'] / scale_factor,
                alpha=0.5,
                edgecolors='white',
                linewidth=1.2,
                label=row['국가']) # 하단 범례용 라벨

    # 🌟 핵심 추가 포인트: 버블 중앙에 국가명 텍스트 달기
    plt.text(x=row['인게이지먼트'],
             y=row['잠재적 노출량'],
             s=row['국가'],
             fontsize=10,
             ha='center', # 가로 기준 중앙 정렬
             va='center', # 세로 기준 중앙 정렬
             fontweight='bold',
             color='black')

# 5. 축 눈금 단위 포맷팅
def format_x(x, pos):
    if x == 0: return '0'
    return f'{int(x / 1000000)}M'

def format_y(y, pos):
    if y == 0: return '0'
    return f'{int(y / 1000000000)}G'

plt.gca().xaxis.set_major_formatter(FuncFormatter(format_x))
plt.gca().yaxis.set_major_formatter(FuncFormatter(format_y))

# 6. 타이틀과 축 라벨 설정
plt.title('국가별 한국 관광 관련 언급 포지셔닝 맵', fontsize=18, fontweight='bold', pad=20)
plt.xlabel('인게이지먼트', fontsize=12)
plt.ylabel('잠재적 노출량', fontsize=12)

# 7. 배경 그리드(격자) 켜기
plt.grid(True, linestyle='-', color='lightgrey', alpha=0.7)

# 8. 범례(Legend) 설정
# 💡 이제 버블 위에 이름이 있으니 하단 범례가 불필요하다면 아래 두 줄을 지우거나 주석(#) 처리하셔도 아주 깔끔합니다!
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15),
           ncol=5, frameon=False, fontsize=11, scatterpoints=1)

# 레이아웃 정리 후 출력
plt.tight_layout()
plt.show()

# 공중화장실

In [ ]:
BASE_DIR = os.getcwd()                     # 현재 작업 중인 기본 폴더 경로
FOLDER_DIR = os.path.join(BASE_DIR, "완전최종_전처리완료_CSV모음")  # 가공 데이터 저장 폴더

DATA_DIR = os.path.join(FOLDER_DIR, "공중화장실.csv")

restroom = pd.read_csv(DATA_DIR)

restroom

In [ ]:
# 1) 비어있는 값 제거
restroom_clean = restroom.dropna(subset=['자치구', 'X좌표 (WGS84)', 'Y좌표 (WGS84)'])

# 2) 띄어쓰기만 있거나 빈 글자('')인 이상한 데이터 제거
# 글자 양옆 공백을 지운(strip) 뒤에 남은 글자가 있는 행만 살립니다.
restroom_clean = restroom_clean[restroom_clean['자치구'].astype(str).str.strip() != '']

restroom_clean

In [ ]:
# 1. 서울시의 대략적인 위경도 정상 범위 (Bounding Box) 설정
# 이 범위를 벗어나면 일단 이상치로 간주합니다.
SEOUL_LAT_MIN, SEOUL_LAT_MAX = 37.4, 37.7   # 위도(Y) 정상 범위
SEOUL_LON_MIN, SEOUL_LON_MAX = 126.7, 127.2 # 경도(X) 정상 범위

print(f"원본 데이터 개수: {len(restroom_clean)}개")

# ==========================================
# [Step 1] 이상치 탐지 및 X-Y 스와프(복구) 시도
# ==========================================
def fix_coordinates(row):
    x = row['X좌표 (WGS84)']
    y = row['Y좌표 (WGS84)']

    # 만약 X좌표 자리에 37점대(위도)가 있고, Y좌표 자리에 127점대(경도)가 있다면?
    # -> 입력자가 X와 Y를 반대로 적은 것이 확실하므로 서로 바꿔서 정상화해 줍니다!
    if (36 < x < 38) and (126 < y < 128):
        return pd.Series([y, x]) # [원래 경도, 원래 위도] 순서로 복구

    # 정상이거나, 아예 알 수 없는 숫자라면 그대로 둡니다 (Step 2에서 처리)
    return pd.Series([x, y])

# apply 함수를 사용해 전체 데이터의 좌표를 점검하고 복구합니다.
restroom_clean[['X좌표 (WGS84)', 'Y좌표 (WGS84)']] = restroom_clean.apply(fix_coordinates, axis=1)

# ==========================================
# [Step 2] 복구 불가능한 진짜 불량 데이터(결측치) 제거
# ==========================================
# 서울시 정상 경계 안에 들어오는 데이터만 필터링해서 남깁니다.
valid_condition = (
    (restroom_clean['Y좌표 (WGS84)'] >= SEOUL_LAT_MIN) & (restroom_clean['Y좌표 (WGS84)'] <= SEOUL_LAT_MAX) &
    (restroom_clean['X좌표 (WGS84)'] >= SEOUL_LON_MIN) & (restroom_clean['X좌표 (WGS84)'] <= SEOUL_LON_MAX)
)

restroom_clean_final = restroom_clean[valid_condition].copy()

# ==========================================
# 결과 확인
# ==========================================
dropped_count = len(restroom_clean) - len(restroom_clean_final)
print(f"삭제된 복구 불가 불량 데이터 개수: {dropped_count}개")
print(f"최종 정제 완료! 남은 깨끗한 데이터 개수: {len(restroom_clean_final)}개")

In [ ]:
# 1. 한글 폰트 설정 (깨짐 방지용)
plt.rc('font', family='Malgun Gothic') # Windows
# plt.rc('font', family='AppleGothic') # Mac
plt.rcParams['axes.unicode_minus'] = False

# 2. 자치구별 화장실 개수 집계 및 정렬
# value_counts()로 개수를 세고, 가로 막대그래프에서 가장 많은 구가 맨 위에 오도록
# sort_values(ascending=True)로 오름차순 정렬을 해줍니다.
restroom_counts = restroom_clean_final['자치구'].value_counts().sort_values(ascending=True)

# 3. 도화지(Figure) 세팅
plt.figure(figsize=(12, 10))

# 4. 가로 막대 그래프 그리기 (Seaborn 팔레트 활용)
# 색상에 그라데이션을 주어 시각적으로 더 깔끔하게 표현합니다.
bars = plt.barh(restroom_counts.index, restroom_counts.values,
                color=sns.color_palette("YlGnBu", len(restroom_counts)))

# 5. 🌟 핵심 포인트: 막대 끝에 정확한 수치 레이블 달기
for bar in bars:
    width = bar.get_width() # 막대의 길이 = 화장실 개수
    plt.text(width + 5, # 막대 끝에서 살짝(5포인트) 오른쪽으로 띄워서 글씨 배치
             bar.get_y() + bar.get_height() / 2, # 막대의 세로 정중앙
             f'{int(width)}개',
             ha='left', va='center', fontsize=11, fontweight='bold', color='black')

# 6. 그래프 꾸미기
plt.title('서울시 자치구별 공중화장실 분포 현황', fontsize=18, fontweight='bold', pad=20)
plt.xlabel('공중화장실 수 (개)', fontsize=13)
plt.ylabel('자치구', fontsize=13)

# X축 눈금에만 세로 보조선(그리드)을 그어 길이를 쉽게 가늠하도록 합니다.
plt.grid(axis='x', linestyle='--', alpha=0.7)

# 레이아웃 깔끔하게 정리 후 출력
plt.tight_layout()
plt.show()

In [ ]:
# 자치구별 화장실 밀도(개수) 집계 데이터프레임 만들기
# Choropleth(단계구분도)에 색상을 칠하기 위해 구별 개수를 셉니다.
restroom_density = restroom_clean_final.groupby('자치구').size().reset_index(name='화장실수')

# 서울 중심부(시청 기준) 좌표로 기본 도화지(지도) 생성
m = folium.Map(location=[37.5665, 126.9780], zoom_start=11)

# ==========================================
# [레이어 1: 배경] 자치구별 밀도 단계구분도 (Choropleth)
# ==========================================
# ⚠️ geo_data에는 이전 모텔 분석 때 쓰셨던 'gdf_map'이나 서울시 GeoJSON 파일을 넣어주세요!
# ⚠️ key_on 값은 사용하시는 GeoJSON 구조(보통 'feature.properties.SIG_KOR_NM' 등)에 맞춰 수정이 필요할 수 있습니다.
folium.Choropleth(
    geo_data=gdf_map, # 서울시 행정구역 경계 데이터 (GeoDataFrame 또는 JSON 경로)
    data=restroom_density,
    columns=['자치구', '화장실수'],
    key_on='feature.properties.SIGNGU_NM', # gdf_map의 구조에 맞게 매칭 (예: feature.properties.name)
    fill_color='YlOrRd', # 노란색 -> 주황색 -> 빨간색 그라데이션
    fill_opacity=0.6,    # 마커가 잘 보이도록 배경은 살짝 투명하게
    line_opacity=0.3,
    legend_name='서울시 자치구별 공중화장실 밀도 (개수)'
).add_to(m)

# ==========================================
# [레이어 2: 점] 공중화장실 실제 위치 파란색 마커
# ==========================================
# iterrows()를 사용해 4천여 개의 데이터를 하나씩 돌면서 지도에 파란 점을 찍습니다.
for idx, row in restroom_clean_final.iterrows():
    folium.CircleMarker(
        location=[row['Y좌표 (WGS84)'], row['X좌표 (WGS84)']], # Folium은 [위도(Y), 경도(X)] 순서입니다!
        radius=2,           # 점 크기 (너무 크면 징그러우니 작게 설정)
        color='darkblue',   # 테두리 색상
        weight=0.5,         # 테두리 두께
        fill=True,
        fill_color='blue',  # 내부 채우기 색상
        fill_opacity=0.7    # 점의 투명도
    ).add_to(m)

# 완성된 지도 출력
m

In [ ]:
# 1. 한글 폰트 설정
plt.rc('font', family='Malgun Gothic') # Windows
# plt.rc('font', family='AppleGothic') # Mac
plt.rcParams['axes.unicode_minus'] = False

# ==========================================
# 2. 데이터 병합 (Merge) 준비
# ==========================================
# 1) 공중화장실 데이터 집계 (restroom_clean_final은 이전 단계에서 이상치가 제거된 깨끗한 데이터)
restroom_cnt = restroom_clean_final.groupby('자치구').size().reset_index(name='화장실수')

# 2) 일반숙박업소 데이터 집계
motel_cnt = motel.groupby('자치구').size().reset_index(name='사업장_개수')

# 3) 두 데이터 병합 (자치구를 기준으로 합치기)
df_merged = pd.merge(restroom_cnt, motel_cnt, on='자치구', how='inner')

# ==========================================
# 3. 포지셔닝 맵(산점도) 그리기
# ==========================================
plt.figure(figsize=(14, 10))

# 산점도 점 찍기 (점 크기는 100, 투명도는 0.7)
sns.scatterplot(data=df_merged, x='사업장_개수', y='화장실수',
                s=150, alpha=0.7, color='dodgerblue', edgecolor='black')

# 각 점 옆에 자치구 이름(텍스트) 달아주기
for i in range(len(df_merged)):
    plt.text(df_merged['사업장_개수'].iloc[i] + 5,
             df_merged['화장실수'].iloc[i],
             df_merged['자치구'].iloc[i],
             fontsize=11, fontweight='bold')

# ==========================================
# 4. 분석의 핵심! 4분면(Quadrant)을 나누는 평균선 긋기
# ==========================================
x_mean = df_merged['사업장_개수'].mean()
y_mean = df_merged['화장실수'].mean()

plt.axvline(x_mean, color='red', linestyle='--', alpha=0.6, linewidth=2) # 세로선 (숙박업소 평균)
plt.axhline(y_mean, color='red', linestyle='--', alpha=0.6, linewidth=2) # 가로선 (화장실 평균)

# 그래프 꾸미기
plt.title('서울시 자치구별 관광 인프라 포지셔닝 맵 (일반숙박업소 vs 화장실)', fontsize=18, fontweight='bold', pad=20)
plt.xlabel('일반숙박업소 수 (개)', fontsize=13)
plt.ylabel('공중화장실 수 (개)', fontsize=13)
plt.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# 1. 한글 폰트 설정
plt.rc('font', family='Malgun Gothic') # Windows
# plt.rc('font', family='AppleGothic') # Mac
plt.rcParams['axes.unicode_minus'] = False

# ==========================================
# 2. 데이터 병합 (Merge) 준비
# ==========================================
# 1) 공중화장실 데이터 집계 (restroom_clean_final은 이전 단계에서 이상치가 제거된 깨끗한 데이터)
#restroom_cnt = restroom_clean_final.groupby('자치구').size().reset_index(name='화장실수')

# 2) 관광숙박업소 데이터 집계
hotel_cnt = hotel.groupby('자치구').size().reset_index(name='사업장_개수')

# 3) 두 데이터 병합 (자치구를 기준으로 합치기)
hr_merged = pd.merge(restroom_cnt, hotel_cnt, on='자치구', how='inner')

# ==========================================
# 3. 포지셔닝 맵(산점도) 그리기
# ==========================================
plt.figure(figsize=(14, 10))

# 산점도 점 찍기 (점 크기는 100, 투명도는 0.7)
sns.scatterplot(data=hr_merged, x='사업장_개수', y='화장실수',
                s=150, alpha=0.7, color='dodgerblue', edgecolor='black')

# 각 점 옆에 자치구 이름(텍스트) 달아주기
for i in range(len(df_merged)):
    plt.text(hr_merged['사업장_개수'].iloc[i] + 5,
             hr_merged['화장실수'].iloc[i],
             hr_merged['자치구'].iloc[i],
             fontsize=11, fontweight='bold')

# ==========================================
# 4. 분석의 핵심! 4분면(Quadrant)을 나누는 평균선 긋기
# ==========================================
x_mean = hr_merged['사업장_개수'].mean()
y_mean = hr_merged['화장실수'].mean()

plt.axvline(x_mean, color='red', linestyle='--', alpha=0.6, linewidth=2) # 세로선 (숙박업소 평균)
plt.axhline(y_mean, color='red', linestyle='--', alpha=0.6, linewidth=2) # 가로선 (화장실 평균)

# 그래프 꾸미기
plt.title('서울시 자치구별 관광 인프라 포지셔닝 맵 (관광숙박업소 vs 화장실)', fontsize=18, fontweight='bold', pad=20)
plt.xlabel('관광숙박업소 수 (개)', fontsize=13)
plt.ylabel('공중화장실 수 (개)', fontsize=13)
plt.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import warnings
import requests
from pathlib import Path

import pandas as pd
import numpy as np
import re
import seaborn as sns

import geopandas as gpd
from pyproj import Transformer
from shapely.geometry import Point

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

from typing import Optional, Tuple, Dict
from datetime import datetime
import folium

warnings.filterwarnings("ignore")
pd.options.display.float_format = '{:.2f}'.format

# Mac용 한글 폰트(AppleGothic) 및 마이너스 깨짐 방지
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

%config InlineBackend.figure_format = 'retina'

# 파일 불러오기

In [ ]:
# 1. 기본 경로 및 폴더 설정
BASE_DIR = os.getcwd()                                               # 현재 작업 중인 기본 폴더 경로
FOLDER_DIR = os.path.join(BASE_DIR, "완전최종_전처리완료_CSV모음")   # 가공 데이터 저장 폴더 (필요시 "병합완료"로 수정)

# ==========================================
# 2. 인프라 및 치안 데이터
# ==========================================
df_police = pd.read_csv(os.path.join(FOLDER_DIR, "치안센터.csv"))
df_rroom = pd.read_csv(os.path.join(FOLDER_DIR, "공중화장실.csv"))
df_cctv_final = pd.read_csv(os.path.join(FOLDER_DIR, "cctv.csv"))

# ==========================================
# 3. 인구 및 상권 데이터
# ==========================================
gdf_subway_final = pd.read_csv(os.path.join(FOLDER_DIR, "지하철_유동인구.csv"))
gdf_time_subway_final = pd.read_csv(os.path.join(FOLDER_DIR, "지하철_시간대별평균유동인구.csv"))
df_commerce_final = pd.read_csv(os.path.join(FOLDER_DIR, "상권_유동인구_직장인구_상주인구_상권변화지표.csv"))

# ==========================================
# 4. 관광 및 명소 데이터
# ==========================================
gdf_landmark_5179 = pd.read_csv(os.path.join(FOLDER_DIR, "서울시_관광_명소_보완.csv"))
gdf_120_place_5179 = pd.read_csv(os.path.join(FOLDER_DIR, "서울시_주요_120장소_목록.csv"))
gdf_park_5179 = pd.read_csv(os.path.join(FOLDER_DIR, "서울시_주요_공원.csv"))
df_festival = pd.read_csv(os.path.join(FOLDER_DIR, "서울시_문화행사정보.csv"))

# ==========================================
# 5. 검색어 및 언급량 (트렌드) 데이터
# ==========================================

df_search_rank = pd.read_csv(os.path.join(FOLDER_DIR, "카테고리별_검색순위.csv"))
df_keyword = pd.read_csv(os.path.join(FOLDER_DIR, "키워드별_언급량.csv"))
df_kor_tour_country_2025 = pd.read_csv(os.path.join(FOLDER_DIR, "한국관광관련_국가별_언급량_인게이지먼트_잠재적_노출량.csv"))
df_kor_tour_mention = pd.read_csv(os.path.join(FOLDER_DIR, "한국관광관련_언급량_인게이지먼트_추이.csv"))

# ==========================================
# 6. 숙박 및 방문객 데이터
# ==========================================
hotel_cleaned_5179 = pd.read_csv(os.path.join(FOLDER_DIR, "서울시_관광숙박업_인허가_정보.csv"))
motel_subset_5179 = pd.read_csv(os.path.join(FOLDER_DIR, "서울시_일반숙박업_인허가_정보.csv"))
native_foreigner = pd.read_csv(os.path.join(FOLDER_DIR, "자치구별_내외국인_방문자_수_추이.csv"))
stay = pd.read_csv(os.path.join(FOLDER_DIR, "자치구별_숙박방문외지인_숙박_체류시간_월별데이터.csv"))
df_airbnb = pd.read_csv(os.path.join(FOLDER_DIR, "에어비앤비.csv"))

# ==========================================
# 7. 교통 데이터
# ==========================================
df_subway_clean = pd.read_csv(os.path.join(FOLDER_DIR, "지하철역사정보.csv"))
df_bus_station_merged = pd.read_csv(os.path.join(FOLDER_DIR, "버스정류소_위치정보.csv"))

In [ ]:
df_search_rank.columns

In [ ]:
# 1. 기본 경로 및 폴더 설정 (상대 경로 방식)
BASE_DIR = os.getcwd()                                               # 현재 작업 중인 기본 폴더 경로
FOLDER_DIR = os.path.join(BASE_DIR, "완전최종_전처리완료_GPKG모음")   # GPKG 데이터 저장 폴더

# ==========================================
# 2. 인프라 및 치안 데이터
# ==========================================
gdf_police = gpd.read_file(os.path.join(FOLDER_DIR, "치안센터.gpkg"))
gdf_rroom = gpd.read_file(os.path.join(FOLDER_DIR, "공중화장실.gpkg"))
gdf_cctv_final = gpd.read_file(os.path.join(FOLDER_DIR, "cctv.gpkg"))

# ==========================================
# 3. 인구 및 상권 데이터
# ==========================================
gdf_subway_final = gpd.read_file(os.path.join(FOLDER_DIR, "지하철_유동인구.gpkg"))
gdf_time_subway_final = gpd.read_file(os.path.join(FOLDER_DIR, "지하철_시간대별평균유동인구.gpkg"))
gdf_commerce_final = gpd.read_file(os.path.join(FOLDER_DIR, "상권_유동인구_직장인구_상주인구_상권변화지표.gpkg"))

# ==========================================
# 4. 관광 및 명소 데이터
# ==========================================
gdf_landmark = gpd.read_file(os.path.join(FOLDER_DIR, "서울시_관광_명소_보완.gpkg"))
gdf_120_place = gpd.read_file(os.path.join(FOLDER_DIR, "서울시_주요_120장소_목록.gpkg"))
gdf_park = gpd.read_file(os.path.join(FOLDER_DIR, "서울시_주요_공원.gpkg"))
gdf_festival = gpd.read_file(os.path.join(FOLDER_DIR, "서울시_문화행사정보.gpkg"))

# ==========================================
# 5. 숙박 데이터
# ==========================================
gdf_hotel = gpd.read_file(os.path.join(FOLDER_DIR, "서울시_관광숙박업_인허가_정보.gpkg"))
gdf_motel = gpd.read_file(os.path.join(FOLDER_DIR, "서울시_일반숙박업_인허가_정보.gpkg"))
gdf_airbnb = gpd.read_file(os.path.join(FOLDER_DIR, "에어비앤비.gpkg"))

# ==========================================
# 6. 교통 데이터
# ==========================================
gdf_subway_clean = gpd.read_file(os.path.join(FOLDER_DIR, "지하철역사정보.gpkg"))
gdf_bus_station = gpd.read_file(os.path.join(FOLDER_DIR, "버스정류소_위치정보.gpkg"))

# 지하철 유동인구

In [ ]:
# 1. 역사명 기준으로 총승하차승객수를 먼저 다 합쳐줍니다 (월별/호선별 데이터 병합)
station_grouped = gdf_subway_final.groupby('역사명')['총승하차승객수'].sum().reset_index()

# 2. 합산된 데이터를 기준으로 다시 상위 15개 역을 추출합니다
top_stations = station_grouped.sort_values(by='총승하차승객수', ascending=False).head(15)

# 3. 그래프 그리기
plt.figure(figsize=(20, 6))
sns.barplot(data=top_stations, x='역사명', y='총승하차승객수', palette='viridis')
plt.title('총 유동인구 상위 15개 지하철역')
plt.ylabel('총 승하차 승객 수')
plt.xlabel('역사명')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# 1. 역별로 승차, 하차 승객수 각각 합산
station_boarding_alighting = gdf_subway_final.groupby('역사명')[['승차총승객수', '하차총승객수']].sum().reset_index()

# 2. 승차 기준 Top 10 추출
top_10_boarding = station_boarding_alighting.sort_values(by='승차총승객수', ascending=False).head(10)

# 3. 하차 기준 Top 10 추출
top_10_alighting = station_boarding_alighting.sort_values(by='하차총승객수', ascending=False).head(10)

# 4. 그래프 그리기 (1행 2열로 나란히 배치)
fig, axes = plt.subplots(1, 2, figsize=(20, 7)) # 가로 20, 세로 7 크기

# [왼쪽 그래프] 승차 승객수 Top 10
sns.barplot(
    data=top_10_boarding,
    x='승차총승객수',
    y='역사명',
    ax=axes[0],
    palette='Blues_r'
)
axes[0].set_title('승차 승객수 Top 10 (출발지 성격)', fontsize=16)
axes[0].set_xlabel('총 승차 승객 수', fontsize=12)
axes[0].set_ylabel('지하철역', fontsize=12)

# [오른쪽 그래프] 하차 승객수 Top 10
sns.barplot(
    data=top_10_alighting,
    x='하차총승객수',
    y='역사명',
    ax=axes[1],
    palette='Reds_r'
)
axes[1].set_title('하차 승객수 Top 10 (목적지 성격)', fontsize=16)
axes[1].set_xlabel('총 하차 승객 수', fontsize=12)
axes[1].set_ylabel('') # y축 라벨 생략 (왼쪽과 겹치므로)

# 5. 전체 타이틀 및 레이아웃 정리
plt.suptitle('지하철 유동인구: 승차 vs 하차 Top 10 비교', fontsize=20, y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# 1. 월별, 역사명별 총승하차승객수 합산 (중복 데이터 방지)
monthly_station_traffic = gdf_subway_final.groupby(['월', '역사명'])['총승하차승객수'].sum().reset_index()

# 2. 월별로 그룹화한 뒤, 승객수 기준으로 내림차순 정렬하여 상위 10개씩만 추출
top_10_per_month = monthly_station_traffic.sort_values(['월', '총승하차승객수'], ascending=[True, False]).groupby('월').head(10)

# 3. 보기 쉽게 '순위' 컬럼 추가
top_10_per_month['순위'] = top_10_per_month.groupby('월').cumcount() + 1

# 4. 피벗 테이블을 활용해 가로축은 '월', 세로축은 '순위'인 깔끔한 표로 변환
ranking_table = top_10_per_month.pivot(index='순위', columns='월', values='역사명')

print("월별 유동인구 Top 10 지하철역 순위표")
display(ranking_table)

In [ ]:
# 1. 역별 총 승차, 하차 승객수 합산
station_flow = gdf_subway_final.groupby('역사명')[['승차총승객수', '하차총승객수']].sum().reset_index()

# 2. '순유입 인구' 파생 변수 생성 (하차총승객수 - 승차총승객수)
station_flow['순유입인구'] = station_flow['하차총승객수'] - station_flow['승차총승객수']

# 3. 순유입 인구가 가장 많은 상위 10개 역 추출 (진짜 핫플레이스)
top_10_net_inflow = station_flow.sort_values(by='순유입인구', ascending=False).head(15)

# 4. 시각화 (바 차트)
plt.figure(figsize=(12, 6))

# 강렬한 색상 팔레트 사용하여 집중도 높이기
sns.barplot(
    data=top_10_net_inflow,
    x='역사명',
    y='순유입인구',
    palette='magma'
)

plt.title('순유입 인구(하차-승차) Top 10 지하철역 (게스트 최종 목적지)', fontsize=16)
plt.ylabel('순유입 승객 수 (명)', fontsize=12)
plt.xlabel('지하철역', fontsize=12)
plt.xticks(rotation=45)

# 기준선 (0) 추가 및 그리드 설정
plt.axhline(0, color='black', linewidth=1)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()

plt.show()

## 상권 유동인구

In [ ]:
# 1. '총_순수방문객_수' 파생변수 생성 (총 유동인구 - (직장인구 + 상주인구))
df_commerce_final['총_순수방문객_수'] = df_commerce_final['총_유동인구_수'] - (df_commerce_final['총_직장_인구_수'] + df_commerce_final['총_상주인구_수'])

# ==============================================================

# 2. 상권명 기준으로 데이터 합산 (분기별 데이터 통합)
grouped_data = df_commerce_final.groupby('상권_코드_명')[['총_유동인구_수', '총_순수방문객_수']].sum().reset_index()

# 3. 각각의 Top 10 데이터 추출
# [1] 총 유동인구 기준 Top 10
top10_total = grouped_data.sort_values(by='총_유동인구_수', ascending=False).head(10)
# [2] 순수 방문객 수 기준 Top 10
top10_pure = grouped_data.sort_values(by='총_순수방문객_수', ascending=False).head(10)

# 4. 2행 1열 서브플롯으로 두 그래프를 위아래로 배치 (세로 길이를 14로 넉넉하게 설정)
fig, axes = plt.subplots(2, 1, figsize=(15, 14))

# [위쪽 그래프] 총 유동인구 Top 10
sns.barplot(data=top10_total, x='상권_코드_명', y='총_유동인구_수', ax=axes[0], palette='Blues_r')
axes[0].set_title('총 유동인구 수 Top 10 상권', fontsize=16)
axes[0].set_ylabel('총 유동인구 수 (명)')
axes[0].set_xlabel('상권명')
axes[0].tick_params(axis='x', rotation=45)

# [아래쪽 그래프] 순수 방문객 수 Top 10
sns.barplot(data=top10_pure, x='상권_코드_명', y='총_순수방문객_수', ax=axes[1], palette='Oranges_r')
axes[1].set_title('순수 방문객 수 Top 10 상권', fontsize=16)
axes[1].set_ylabel('순수 방문객 수 (명)')
axes[1].set_xlabel('상권명')
axes[1].tick_params(axis='x', rotation=45)

# 레이아웃 여백 자동 조정 및 출력
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 1. 데이터 집계 및 포맷팅
# ==========================================
quarterly_trend = df_commerce_final.groupby('기준_년분기_코드')[['총_유동인구_수', '총_순수방문객_수']].sum().reset_index()
quarterly_trend['분기_포맷'] = quarterly_trend['기준_년분기_코드'].astype(str).str[:4] + '년 ' + quarterly_trend['기준_년분기_코드'].astype(str).str[4] + 'Q'

# ==========================================
# 2. 막대그래프 + 선 그래프 (콤보 차트) 시각화
# ==========================================
plt.figure(figsize=(14, 6))

# [1] 막대 그래프: 총 유동인구 수
sns.barplot(
    data=quarterly_trend,
    x='분기_포맷',
    y='총_유동인구_수',
    color='#B0BEC5',
    alpha=0.6, # 조금 더 투명하게 해서 선 그래프를 강조
    label='총 유동인구 수'
)

# [2] 선 그래프: 총 순수 방문객 수
sns.lineplot(
    data=quarterly_trend,
    x='분기_포맷',
    y='총_순수방문객_수',
    marker='o',
    color='#FF7043',
    linewidth=3, # 선을 조금 더 굵고 선명하게
    markersize=10,
    label='총 순수 방문객 수'
)

# ==========================================
# 3. 🎨 Y축 여백(Headroom) 넉넉하게 주기
# ==========================================
# 유동인구 최댓값을 구한 뒤, 그 값의 1.3배(30% 여백)를 Y축의 끝으로 설정합니다.
max_y_value = quarterly_trend['총_유동인구_수'].max()
plt.ylim(0, max_y_value * 1.3)

# ==========================================
# 4. 디자인 및 레이아웃 설정
# ==========================================
plt.title('서울시 상권 유동인구 및 순수 방문객 수 분기별 변화 추이', fontsize=18, fontweight='bold', pad=20)
plt.xlabel('분기', fontsize=12)
plt.ylabel('인구 수 (명)', fontsize=12)
plt.xticks(rotation=45)

# Y축 그리드선 추가 (점선으로 은은하게)
plt.grid(True, axis='y', alpha=0.3, linestyle='--')

# 범례 위치를 그래프 오른쪽 상단 바깥으로 깔끔하게 이동
plt.legend(title='인구 유형', fontsize=11, loc='upper left', bbox_to_anchor=(1, 1))

# 숫자(Y축)가 지수형태(1e8 등)로 나오는 것을 방지하고 일반 숫자로 표시
plt.ticklabel_format(axis='y', style='plain')
import matplotlib.ticker as ticker
plt.gca().yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: format(int(x), ','))) # 천 단위 콤마 추가

plt.tight_layout()
plt.show()

In [ ]:
#야간 vs 주간
# 1. 계산에 사용할 시간대별 컬럼 리스트 정의
daytime_cols = ['시간대_06_11_유동인구_수', '시간대_11_14_유동인구_수', '시간대_14_17_유동인구_수']
nighttime_cols = ['시간대_17_21_유동인구_수', '시간대_21_24_유동인구_수', '시간대_00_06_유동인구_수']

# 데이터 숫자형 변환 (오류 발생 시 NaN 처리 후 0으로 채움)
cols_to_convert = daytime_cols + nighttime_cols
df_commerce_final[cols_to_convert] = df_commerce_final[cols_to_convert].apply(pd.to_numeric, errors='coerce').fillna(0)

# 2. 로우(분기)별 주간/야간 유동인구 합계 및 비중 계산 (groupby 없이 바로 계산)
df_commerce_final['주간_유동인구_수'] = df_commerce_final[daytime_cols].sum(axis=1)
df_commerce_final['야간_유동인구_수'] = df_commerce_final[nighttime_cols].sum(axis=1)
df_commerce_final['총_유동인구_합계'] = df_commerce_final['주간_유동인구_수'] + df_commerce_final['야간_유동인구_수']

# 0으로 나누기 방지
df_commerce_final['주간_유동인구_비중'] = np.where(df_commerce_final['총_유동인구_합계'] == 0, 0,
                                            (df_commerce_final['주간_유동인구_수'] / df_commerce_final['총_유동인구_합계']) * 100)
df_commerce_final['야간_유동인구_비중'] = np.where(df_commerce_final['총_유동인구_합계'] == 0, 0,
                                            (df_commerce_final['야간_유동인구_수'] / df_commerce_final['총_유동인구_합계']) * 100)

# 3. 주간/야간/종일 활성화 유형 라벨링 적용 (비중 차이 10%p 이하 기준)
conditions = [
    abs(df_commerce_final['주간_유동인구_비중'] - df_commerce_final['야간_유동인구_비중']) <= 10,
    df_commerce_final['주간_유동인구_비중'] > df_commerce_final['야간_유동인구_비중']
]
choices = ['종일 활성화', '주간 활성화']

df_commerce_final['시간대별_활성화_유형'] = np.select(conditions, choices, default='야간 활성화')
df_commerce_final.loc[df_commerce_final['총_유동인구_합계'] == 0, '시간대별_활성화_유형'] = '데이터 없음'


# 4. 🌟 자치구별/상권별/분기별 피벗 테이블 생성 (시간대별 활성화 유형)
quarter_col = '기준_년분기_코드'

try:
    # 1. 다른 피벗 테이블과 겹치지 않도록 'pivot_time_activation'으로 변수명 변경
    pivot_time_activation = df_commerce_final.pivot_table(
        index=['자치구', '상권_구분_코드_명', '상권_코드_명'], # 행: 자치구 > 상권구분 > 상권명
        columns=quarter_col,                 # 열: 202501, 202502, 202503 등
        values='시간대별_활성화_유형',          # 값: 활성화 유형 텍스트
        aggfunc='first'
    )

    # 2. 💡 핵심: .reset_index()를 지웠습니다!
    # 그래야 '강남구'가 맨 위에 한 번만 표시되고, 그 아래 상권들이 예쁘게 그룹화되어 묶여 보입니다.

    print("📊 자치구 내 상권별 분기 변화 요약:")
    display(pivot_time_activation.head())

except KeyError as e:
    print(f"오류: 데이터프레임에 필요한 컬럼이 없습니다. {e}")

In [ ]:
pivot_time_activation.head(20)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# 1. 인덱스 초기화 (자치구별 그룹화를 위해)
df_reset = pivot_time_activation.reset_index()

# 2. 기준 분기 컬럼 자동 탐색 (KeyError 방지)
if 20253 in df_reset.columns:
    target_col = 20253
elif '20253' in df_reset.columns:
    target_col = '20253'
# 피벗테이블 멀티인덱스 컬럼 구조일 경우를 대비
elif ('기준_년분기_코드', 20253) in df_reset.columns:
    target_col = ('기준_년분기_코드', 20253)
elif ('기준_년분기_코드', '20253') in df_reset.columns:
    target_col = ('기준_년분기_코드', '20253')
else:
    target_col = df_reset.columns[-1] # 못 찾으면 가장 마지막 컬럼 사용

# 3. 자치구 및 상권 유형별 개수 집계
count_df = df_reset.groupby(['자치구', target_col]).size().unstack(fill_value=0)

# 4. 백분율(%) 비율 계산 (각 행의 합으로 나누기)
ratio_df = count_df.div(count_df.sum(axis=1), axis=0) * 100

# 5. ⭐ '주간 활성화' 비율이 높은 순(내림차순)으로 정렬되도록 수정 ⭐
if '주간 활성화' in ratio_df.columns:
    ratio_df = ratio_df.sort_values(by='주간 활성화', ascending=False)

# 6. 대비 및 혼합 색상 지정
color_mapping = {
    '주간 활성화': '#FFD700',  # 노랑 (주간)
    '야간 활성화': '#0055A4',  # 파랑 (야간)
    '종일 활성화': '#2CA02C'   # 초록 (종일)
}

# 데이터프레임의 컬럼(활성화 유형) 순서에 맞춰 색상 리스트 생성
plot_colors = [color_mapping.get(col, '#CCCCCC') for col in ratio_df.columns]

# 7. 누적 비율 막대 그래프 그리기 (stacked=True)
ax = ratio_df.plot(kind='bar', stacked=True, figsize=(15, 6), color=plot_colors, width=0.8, edgecolor='white')

# ==========================================
# ⭐ 막대 그래프 내부에 레이블(비율) 표시 추가 ⭐
# ==========================================
for c in ax.containers:
    # 각 막대 구간의 높이(비율값)를 가져와서 3% 이상인 경우에만 소수점 첫째 자리까지 표시
    # (비율이 너무 작으면 글자가 겹치기 때문. 3.0 숫자를 원하시는 대로 조절하세요)
    labels = [f'{v.get_height():.1f}%' if v.get_height() >= 3.0 else '' for v in c]

    # label_type='center'로 설정하여 각 누적 구간의 가운데에 텍스트가 오게 함
    ax.bar_label(c, labels=labels, label_type='center', fontsize=9, color='black', fontweight='bold')
# ==========================================

# 그래프 꾸미기
plt.title('자치구별 활성화 상권 누적 비율 (2025년 3분기)', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('자치구', fontsize=12)
plt.ylabel('비율 (%)', fontsize=12)
plt.xticks(rotation=45)

# y축을 퍼센트(%) 형식으로 표시
ax.yaxis.set_major_formatter(PercentFormatter())

# 범례를 그래프 우측 바깥으로 이동
plt.legend(title='활성화 상권 유형', bbox_to_anchor=(1.01, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
#평일 vs 주말
# 1. 요일별 유동인구 컬럼 리스트
days = [
    '월요일_유동인구_수', '화요일_유동인구_수', '수요일_유동인구_수',
    '목요일_유동인구_수', '금요일_유동인구_수', '토요일_유동인구_수', '일요일_유동인구_수'
]

# 2. 데이터 타입이 문자열(str)일 수 있으므로 안전하게 숫자형(float/int)으로 변환
# errors='coerce'를 통해 숫자로 바꿀 수 없는 값은 NaN으로 만들고, fillna(0)으로 0 처리합니다.
for day in days:
    df_commerce_final[day] = pd.to_numeric(df_commerce_final[day], errors='coerce').fillna(0)

# 3. 평일 유동인구 일평균 계산 (5일)
df_commerce_final['평일_일평균_유동인구'] = (
    df_commerce_final['월요일_유동인구_수'] +
    df_commerce_final['화요일_유동인구_수'] +
    df_commerce_final['수요일_유동인구_수'] +
    df_commerce_final['목요일_유동인구_수'] +
    df_commerce_final['금요일_유동인구_수']
) / 5

# 4. 주말 유동인구 일평균 계산 (2일)
df_commerce_final['주말_일평균_유동인구'] = (
    df_commerce_final['토요일_유동인구_수'] +
    df_commerce_final['일요일_유동인구_수']
) / 2

# 5. 주말 일평균이 평일 일평균보다 크면 '주말 상권', 아니면 '평일 상권'으로 분류
df_commerce_final['상권_유형'] = np.where(
    df_commerce_final['주말_일평균_유동인구'] > df_commerce_final['평일_일평균_유동인구'],
    '주말 상권',
    '평일 상권'
)

# 6. 결과 확인 (분기, 상권코드, 상권명, 평일/주말 일평균, 상권유형)
# 가지고 계신 데이터에 '기준_년분기_코드' 컬럼이 있다면 포함해서 확인하세요!
result_cols = ['상권_코드', '상권_코드_명', '평일_일평균_유동인구', '주말_일평균_유동인구', '상권_유형']

# 데이터에 '기준_년분기_코드'가 있다면 리스트 맨 앞에 추가
if '기준_년분기_코드' in df_commerce_final.columns:
    result_cols.insert(0, '기준_년분기_코드')

In [ ]:
# 피벗 테이블 생성
pivot_df = df_commerce_final.pivot_table(
    index=['자치구', '상권_구분_코드_명', '상권_코드_명'], # 행으로 들어갈 기준들
    columns='기준_년분기_코드',                        # 열로 들어갈 기준 (분기)
    values='상권_유형',                                 # 표 안에 채워넣을 값 (텍스트 컬럼 이름 입력)
    aggfunc='first'                                     # 텍스트 데이터이므로 첫 번째 값을 그대로 가져옴
)

# 데이터가 없는 분기(NaN)가 있을 경우 보기 좋게 '-' 등으로 채워주기 (선택 사항)
pivot_df = pivot_df.fillna('-')

# 결과 확인
pivot_df.head(20)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# 1. 인덱스 초기화 (요청하신 pivot_df 사용)
df_reset = pivot_df.reset_index()

# 2. 기준 분기 컬럼 자동 탐색
if 20253 in df_reset.columns:
    target_col = 20253
elif '20253' in df_reset.columns:
    target_col = '20253'
# 피벗테이블 멀티인덱스 컬럼 구조일 경우를 대비
elif ('기준_년분기_코드', 20253) in df_reset.columns:
    target_col = ('기준_년분기_코드', 20253)
elif ('기준_년분기_코드', '20253') in df_reset.columns:
    target_col = ('기준_년분기_코드', '20253')
else:
    target_col = df_reset.columns[-1]

# ⭐ 3. '-' (결측/미분류) 데이터 제외하기 ⭐
df_filtered = df_reset[df_reset[target_col] != '-']

# 4. 자치구 및 상권 유형별 개수 집계 (필터링된 데이터 사용)
count_df = df_filtered.groupby(['자치구', target_col]).size().unstack(fill_value=0)

# 5. 백분율(%) 비율 계산 (각 행의 합으로 나누기)
ratio_df = count_df.div(count_df.sum(axis=1), axis=0) * 100

# 6. '평일 상권' 비율이 높은 순(내림차순)으로 정렬
if '평일 상권' in ratio_df.columns:
    ratio_df = ratio_df.sort_values(by='평일 상권', ascending=False)

# 7. 대비되는 색상 지정
color_mapping = {
    '평일 상권': '#FF9900',  # 오렌지 (평일)
    '주말 상권': '#0055A4'   # 파랑 (주말)
}

# 데이터프레임의 컬럼 순서에 맞춰 색상 리스트 생성
plot_colors = [color_mapping.get(col, '#CCCCCC') for col in ratio_df.columns]

# 8. 누적 비율 막대 그래프 그리기
ax = ratio_df.plot(kind='bar', stacked=True, figsize=(15, 6), color=plot_colors, width=0.8, edgecolor='white')

# ==========================================
# ⭐ 막대 그래프 내부에 레이블(비율) 표시 추가 ⭐
# ==========================================
for c in ax.containers:
    # 비율이 3% 이상인 경우에만 소수점 첫째 자리까지 표시 (겹침 방지)
    labels = [f'{v.get_height():.1f}%' if v.get_height() >= 3.0 else '' for v in c]

    # label_type='center'로 누적 막대의 중앙에 텍스트 배치
    ax.bar_label(c, labels=labels, label_type='center', fontsize=9, color='white', fontweight='bold')
# ==========================================

# 그래프 꾸미기
plt.title('자치구별 평일/주말 상권 누적 비율 (2025년 3분기)', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('자치구', fontsize=12)
plt.ylabel('비율 (%)', fontsize=12)
plt.xticks(rotation=45)

# y축을 퍼센트(%) 형식으로 표시
ax.yaxis.set_major_formatter(PercentFormatter())

# 범례를 그래프 우측 바깥으로 이동
plt.legend(title='상권 유형', bbox_to_anchor=(1.01, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 2. 최신 분기 데이터 추출 (상권코드별 중복 제거)
df_latest = df_commerce_final.sort_values(by='기준_년분기_코드', ascending=False) \
                             .drop_duplicates(subset='상권_코드')

# 3. 자치구별 상권 변화 지표 비율 계산
cross_tab_ratio = pd.crosstab(df_latest['자치구'],
                              df_latest['상권_변화_지표_명'],
                              normalize='index') * 100

# 4. ★ '상권확장' 비중이 높은 순서대로 자치구 정렬 ★
# 내림차순(ascending=False)으로 정렬하여 왼쪽일수록 좋은 상권 비중이 높게 표시됩니다.
cross_tab_ratio = cross_tab_ratio.sort_values(by='상권확장', ascending=False)

# 5. 시각화
fig, ax = plt.subplots(figsize=(16, 9))

# 색상 조합 (상권확장과 다이나믹이 잘 보이도록 설정)
cross_tab_ratio.plot(kind='bar',
                     stacked=True,
                     ax=ax,
                     colormap='RdYlBu_r', # 성장(청색계열)과 정체(적색계열) 대비가 좋은 컬러맵
                     edgecolor='white',
                     width=0.7)

# ==========================================
# ⭐ 막대 그래프 내부에 레이블(비율) 표시 추가 ⭐
# ==========================================
for c in ax.containers:
    # 비율이 3% 이상인 경우에만 소수점 첫째 자리까지 표시 (작은 비중은 텍스트 겹침 방지)
    labels = [f'{v.get_height():.1f}%' if v.get_height() >= 3.0 else '' for v in c]

    # label_type='center'로 누적 막대의 중앙에 텍스트 배치
    ax.bar_label(c, labels=labels, label_type='center', fontsize=9, color='black', fontweight='bold')
# ==========================================

# 그래프 꾸미기
plt.title('자치구별 상권 성장성 순위 (상권확장 비중 기준)', fontsize=22, pad=30, fontweight='bold')
plt.xlabel('자치구명', fontsize=14)
plt.ylabel('비중 (%)', fontsize=14)
plt.xticks(rotation=45)
plt.ylim(0, 100)
plt.grid(axis='y', linestyle='--', alpha=0.4)

# 범례 설정
plt.legend(title='상권 변화 지표', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# -------------------------------------------------------------------
# 데이터 전처리
# -------------------------------------------------------------------

# 2. 중복 데이터 처리
# 기준_년분기_코드를 내림차순 정렬하여 가장 최근 분기 데이터가 위로 오게 한 뒤,
# 상권 고유 식별자(상권_코드) 기준으로 중복을 제거하여 최신 데이터만 남깁니다.
df_unique = df_commerce_final.sort_values(by='기준_년분기_코드', ascending=False)
df_unique = df_unique.drop_duplicates(subset=['상권_코드']) # 실제 상권 식별 컬럼명에 맞게 확인

# 3. 자치구별 상권 유형 빈도수 계산 (교차표 생성)
# 알려주신 대로 '자치구' 컬럼을 사용합니다.
cross_tab = pd.crosstab(df_unique['자치구'], df_unique['상권_구분_코드_명'])

# 4. 누적 비율(%)로 변환
# 각 자치구(행)의 총합으로 나누어 비율을 구하고 100을 곱합니다.
cross_tab_prop = cross_tab.div(cross_tab.sum(axis=1), axis=0) * 100

# -------------------------------------------------------------------
# 시각화 (누적 비율 막대 그래프)
# -------------------------------------------------------------------

# 5. 그래프 그리기
fig, ax = plt.subplots(figsize=(12, 8))
cross_tab_prop.plot(kind='bar', stacked=True, ax=ax, colormap='Set3', edgecolor='white')

# ==========================================
# ⭐ 막대 그래프 내부에 레이블(비율) 표시 추가 ⭐
# ==========================================
for c in ax.containers:
    # 비율이 3% 이상인 경우에만 소수점 첫째 자리까지 표시 (작은 비중은 텍스트 겹침 방지)
    labels = [f'{v.get_height():.1f}%' if v.get_height() >= 3.0 else '' for v in c]

    # Set3 컬러맵은 밝은 파스텔톤이므로 검은색(black) 글씨가 가장 잘 보입니다.
    ax.bar_label(c, labels=labels, label_type='center', fontsize=9, color='black', fontweight='bold')
# ==========================================

# 6. 그래프 꾸미기
plt.title('자치구별 상권 유형 누적 비율 막대그래프', fontsize=16, pad=15)
plt.xlabel('자치구', fontsize=12)
plt.ylabel('상권 유형 비율 (%)', fontsize=12)

# 범례 위치 조정 (그래프 밖으로 빼기)
plt.legend(title='상권_구분_코드_명', bbox_to_anchor=(1.02, 1), loc='upper left')

# X축 라벨 회전 (자치구 이름이 겹치지 않게)
plt.xticks(rotation=45)

# 레이아웃 여백 자동 조정 (범례나 라벨이 잘리지 않게 함)
plt.tight_layout()

# 그래프 출력
plt.show()

In [ ]:
df_commerce_final.columns

In [ ]:
# 1. 기본 경로 설정
BASE_DIR = os.getcwd()                     # 현재 작업 중인 기본 폴더 경로
FOLDER_DIR = os.path.join(BASE_DIR, "병합완료")  # 가공 데이터 저장 폴더

# -----------------------------------------------------------------
# 2. 상권 데이터 3종 경로 설정 및 불러오기 (cp949 인코딩 반영)
# -----------------------------------------------------------------
# 직장인구
DATA_DIR_A = os.path.join(FOLDER_DIR, "서울시 상권분석서비스(직장인구-상권배후지).csv")
df_a = pd.read_csv(DATA_DIR_A, encoding='cp949')

# 상주인구
DATA_DIR_B = os.path.join(FOLDER_DIR, "서울시 상권분석서비스(상주인구-상권배후지).csv")
df_b = pd.read_csv(DATA_DIR_B, encoding='cp949')

# 길단위 유동인구
DATA_DIR_C = os.path.join(FOLDER_DIR, "서울시 상권분석서비스(길단위인구-상권배후지).csv")
df_c = pd.read_csv(DATA_DIR_C, encoding='cp949')

In [ ]:
df_a.columns

In [ ]:
# 1. 남길 공통 컬럼과 총 직장인구수 컬럼 지정
target_columns = [
    '기준_년분기_코드',
    '상권_구분_코드',
    '상권_구분_코드_명',
    '상권배후지_코드',
    '상권배후지_코드_명',
    '총_직장_인구_수'
]

# 2. 기존 데이터프레임에 덮어쓰기 (새 변수 생성 X)
df_a = df_a[target_columns]

In [ ]:
# 1. 남길 공통 컬럼과 '총_상주인구_수'만 지정 (가구 수 제외)
target_columns_b = [
    '기준_년분기_코드',
    '상권_구분_코드',
    '상권_구분_코드_명',
    '상권배후지_코드',
    '상권배후지_코드_명',
    '총_상주인구_수'
]

# 2. 기존 데이터프레임(df_b)에 덮어쓰기
df_b = df_b[target_columns_b]

In [ ]:
# 1. 남길 공통 컬럼과 '총_유동인구_수'만 지정
target_columns_c = [
    '기준_년분기_코드',
    '상권_구분_코드',
    '상권_구분_코드_명',
    '상권배후지_코드',
    '상권배후지_코드_명',
    '총_유동인구_수'
]

# 2. 기존 데이터프레임(예: df_c)에 덮어쓰기
# (만약 변수명이 df_commerce 라면 df_commerce = df_commerce[target_columns_c] 로 수정해주세요)
df_c = df_c[target_columns_c]

In [ ]:
from functools import reduce

# 1. 병합의 기준이 되는 공통 컬럼 리스트
common_cols = [
    '기준_년분기_코드',
    '상권_구분_코드',
    '상권_구분_코드_명',
    '상권배후지_코드',
    '상권배후지_코드_명'
]


# 합칠 데이터프레임들을 리스트로 묶어줍니다.
dataframes = [df_a, df_b, df_c]

# reduce를 사용해 순차적으로 모두 병합
df_final = reduce(lambda left, right: pd.merge(left, right, on=common_cols, how='inner'), dataframes).fillna(0)

In [ ]:
# 1. 계산을 위해 숫자형으로 안전하게 변환 (결측치는 0으로)
numeric_cols = ['총_직장_인구_수', '총_상주인구_수', '총_유동인구_수']
for col in numeric_cols:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce').fillna(0)

# 2. 총 유동인구수에서 상주인구와 직장인구를 뺀 '순수_유동인구_수' 계산
# (단, 빼기 결과가 음수가 나오면 0으로 처리하도록 clip(lower=0) 적용)
df_final['순수_유동인구_수'] = (
    df_final['총_유동인구_수'] - df_final['총_상주인구_수'] - df_final['총_직장_인구_수']
).clip(lower=0)

# 3. 3가지 지표(직장, 상주, 순수 유동)의 전체 상권 대비 백분위 랭킹(0~1) 계산
df_final['직장_랭킹'] = df_final['총_직장_인구_수'].rank(pct=True)
df_final['상주_랭킹'] = df_final['총_상주인구_수'].rank(pct=True)
df_final['유동_랭킹'] = df_final['순수_유동인구_수'].rank(pct=True) # 총 유동인구가 아닌 순수 유동인구로 랭킹 계산

# 4. 세 가지 랭킹 중 가장 수치가 높은 항목을 추출하기 위해 랭킹 데이터만 분리
rank_df = df_final[['직장_랭킹', '상주_랭킹', '유동_랭킹']]

# 각 랭킹 컬럼명에 따른 최종 상권 유형 라벨링
type_mapping = {
    '직장_랭킹': '오피스/직장형',
    '상주_랭킹': '주거지/동네형',
    '유동_랭킹': '발달/번화가형 (순수 유동인구 중심)'
}

# 행별로 랭킹이 가장 높은(idxmax) 컬럼을 찾아서 매핑 딕셔너리로 유형 부여
df_final['상권배후지_유형'] = rank_df.idxmax(axis=1).map(type_mapping)

# 5. 불필요한 랭킹 컬럼 삭제 및 결과 확인
df_result = df_final.drop(columns=['직장_랭킹', '상주_랭킹', '유동_랭킹'])

df_result.head(10)

In [ ]:
# 1. 저장할 폴더 및 파일 경로 설정
folder_path = '병합완료'
file_path = f'{folder_path}/상권배후지_유동인구.csv'

df_result.to_csv(file_path, index=False, encoding='utf-8-sig')

In [ ]:
folder_path = '병합완료'
file_path = f'{folder_path}/상권_유동인구_직장인구_상주인구_상권변화지표_추가.csv'

df_commerce_final.to_csv(file_path, index=False, encoding='utf-8-sig')

In [ ]:
df_commerce_final.columns

In [ ]:
df_commerce_final.columns

In [ ]:
# 1. 현재 실행 중인 파이썬 스크립트(또는 주피터 노트북)의 위치를 가져옵니다.
current_dir = os.getcwd()

# 2. os.path.join을 사용하여 운영체제에 맞는 파일 경로를 동적으로 생성합니다.
# (Mac의 '/'와 Windows의 '\' 차이를 자동으로 처리해 줍니다.)
file_path = os.path.join(current_dir, '병합완료', '서울시 상권분석서비스(영역-상권)', '서울시 상권분석서비스(영역-상권).shp')

# 3. 설정한 변수명(df_h)으로 데이터를 불러옵니다.
df_h = gpd.read_file(file_path)

# df_h의 'TRDAR_CD'와 'TRDAR_CD_N' 컬럼명을 각각 '상권_코드', '상권_코드_명'으로 변경합니다.
df_h.rename(columns={
    'TRDAR_CD': '상권_코드',
    'TRDAR_CD_N': '상권_코드_명'
}, inplace=True)

In [ ]:
# 1. '상권_코드' 컬럼의 데이터 타입을 문자열(str)로 통일합니다.
df_commerce_final['상권_코드'] = df_commerce_final['상권_코드'].astype(str)
df_h['상권_코드'] = df_h['상권_코드'].astype(str)

# 2. 가져올 컬럼 지정
columns_to_merge = ['상권_코드', '상권_코드_명', 'ADSTRD_CD', 'ADSTRD_CD_']

# 3. 데이터 병합 다시 실행
df_commerce_final = pd.merge(
    df_commerce_final,
    df_h[columns_to_merge],
    on=['상권_코드', '상권_코드_명'],
    how='left'
)

In [ ]:
# 'ADSTRD_CD'와 'ADSTRD_CD_' 컬럼명을 각각 '행정구_코드', '행정구'로 변경합니다.
df_commerce_final.rename(columns={
    'ADSTRD_CD': '행정동_코드',
    'ADSTRD_CD_': '행정동'
}, inplace=True)

In [ ]:
df_commerce_final.columns

## 방문자수

In [ ]:
# 데이터프레임의 실제 컬럼명('외지인방문자수', '외국인방문자수')에 맞게 이름은 살짝 수정해 주세요.
native_foreigner['게스트수요'] = native_foreigner['외지인방문자수'] + native_foreigner['외국인방문자수']


# ==========================================
# 주제 1: 게스트 수요 규모 Top 10 시각화
# ==========================================
# 자치구별 게스트 수요 합계 계산
guest_volume = native_foreigner.groupby('자치구')['게스트수요'].sum().reset_index()

# 게스트 수요가 가장 많은 상위 10개 자치구 추출
top10_volume = guest_volume.sort_values(by='게스트수요', ascending=False).head(10)

# 시각화 (도화지 생성)
fig, ax = plt.subplots(1, 1, figsize=(10, 7))

sns.barplot(
    data=top10_volume,
    x='게스트수요',
    y='자치구',
    ax=ax,
    palette='Purples_r'
)

# 그래프 제목 및 축 이름 설정
ax.set_title('총 방문자(외지인+외국인) 상위 10개 자치구', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('총 방문자 수 (명)', fontsize=12)
ax.set_ylabel('자치구', fontsize=12)

# X축 숫자 단위 포맷팅 (예: 1000000 -> 1,000,000)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, pos: f'{int(x):,}'))

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 주제 3: 타겟 고객층 성향 (외국인 타겟 비중이 높은 자치구 Top 10)
# ==========================================

# 1. 외국인 및 내국인 비중 계산
native_foreigner['외국인타겟비중(%)'] = (native_foreigner['외국인방문자수'] / native_foreigner['게스트수요']) * 100
native_foreigner['내국인타겟비중(%)'] = 100 - native_foreigner['외국인타겟비중(%)']

# 2. 자치구별로 외국인/내국인 비중 평균 계산
target_mean = native_foreigner.groupby('자치구')[['외국인타겟비중(%)', '내국인타겟비중(%)']].mean().reset_index()

# 3. 외국인 비중이 '높은' 순으로 정렬하여 상위 10개 추출
top10_foreign_target = target_mean.sort_values(by='외국인타겟비중(%)', ascending=False).head(10)

# 4. 시각화를 위해 인덱스를 '자치구'로 설정
top10_foreign_target.set_index('자치구', inplace=True)

# 시각화 (Stacked Bar Chart)
# 외국인(#FFB347)을 아래에, 내국인(#AEC6CF)을 위에 쌓음
ax = top10_foreign_target[['외국인타겟비중(%)', '내국인타겟비중(%)']].plot(
    kind='bar',
    stacked=True,
    figsize=(12, 6),
    color=['#FFB347', '#AEC6CF'],
    width=0.7,
    edgecolor='white'
)

plt.title('외국인 타겟 호스팅 최적 지역 Top 10 (외국인 vs 내국인 비율)', fontsize=16, fontweight='bold', pad=15)
plt.ylabel('게스트 구성 비율 (%)', fontsize=12)
plt.xlabel('자치구', fontsize=12)
plt.xticks(rotation=45, fontsize=11)
plt.legend(['외국인 비율(%)', '내국인 비율(%)'], loc='upper left', bbox_to_anchor=(1, 1))

for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy()
    if height > 5:
        ax.text(x + width/2, y + height/2, f'{height:.1f}%', ha='center', va='center', color='black', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


# ==========================================
# 주제 4: 내국인 타겟 비중이 높은 자치구 Top 10
# ==========================================

# 1. 자치구별로 내/외국인 비중 평균 계산 (기존 계산된 데이터 활용)
target_mean_native = native_foreigner.groupby('자치구')[['내국인타겟비중(%)', '외국인타겟비중(%)']].mean().reset_index()

# 2. 내국인 비중이 '높은' 순으로 정렬하여 상위 10개 추출
top10_native_target = target_mean_native.sort_values(by='내국인타겟비중(%)', ascending=False).head(10)

# 3. 시각화를 위해 인덱스를 '자치구'로 설정
top10_native_target.set_index('자치구', inplace=True)

# 시각화 (Stacked Bar Chart)
# 내국인(#AEC6CF)을 아래에, 외국인(#FFB347)을 위에 쌓음
# 리스트 순서를 ['내국인', '외국인']으로 잡았으므로 색상도 [#AEC6CF, #FFB347] 순서로 적용
ax = top10_native_target[['내국인타겟비중(%)', '외국인타겟비중(%)']].plot(
    kind='bar',
    stacked=True,
    figsize=(12, 6),
    color=['#AEC6CF', '#FFB347'], # 👈 색상 순서를 내국인(파랑), 외국인(주황)으로 매칭
    width=0.7,
    edgecolor='white'
)

plt.title('내국인 타겟 호스팅 최적 지역 Top 10 (내국인 vs 외국인 비율)', fontsize=16, fontweight='bold', pad=15)
plt.ylabel('게스트 구성 비율 (%)', fontsize=12)
plt.xlabel('자치구', fontsize=12)
plt.xticks(rotation=45, fontsize=11)
plt.legend(['내국인 비율(%)', '외국인 비율(%)'], loc='upper left', bbox_to_anchor=(1, 1))

for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy()
    if height > 5:
        ax.text(x + width/2, y + height/2, f'{height:.1f}%', ha='center', va='center', color='black', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
display(df_search_rank.head(5))
display(df_keyword.head(5))


In [ ]:
display(gdf_landmark.head(5))
display(gdf_120_place.head(5))
display(gdf_park.head(5))
display(gdf_festival.head(5))

In [ ]:
df_search_rank.columns

In [ ]:
gdf_landmark.columns

In [ ]:
gdf_120_place.columns

In [ ]:
gdf_120_place['CATEGORY'].unique()

# 카테고리 검색

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker

# ==============================================================
# 데이터 준비 (기존 코드와 동일)
# ==============================================================

# [1] 인기 검색 '중분류 카테고리' (전체 데이터)
top_mid_categories = df_search_rank.groupby('중분류 카테고리')['검색건수'].sum().reset_index()
top_mid_categories = top_mid_categories.sort_values(by='검색건수', ascending=False).head(10)

# [2] 인기 검색 개별 '관광지명' (교통시설 제외)
df_filtered_places = df_search_rank[df_search_rank['소분류 카테고리'] != '교통시설']
top_places = df_filtered_places.groupby('관광지명')['검색건수'].sum().reset_index()
top_places = top_places.sort_values(by='검색건수', ascending=False).head(10)

# [3] 인기 검색 '교통시설' (교통시설만 포함)
df_transport = df_search_rank[df_search_rank['소분류 카테고리'] == '교통시설']
top_transport = df_transport.groupby('관광지명')['검색건수'].sum().reset_index()
top_transport = top_transport.sort_values(by='검색건수', ascending=False).head(10)
# 축 이름을 '교통시설명'으로 변경
top_transport.rename(columns={'관광지명': '교통시설명'}, inplace=True)

# Y축 포맷터 (천 단위 콤마)
formatter = ticker.FuncFormatter(lambda x, p: format(int(x), ','))


# ==============================================================
# 1. 첫 번째 그래프: 중분류 카테고리
# ==============================================================
fig1, ax1 = plt.subplots(figsize=(12, 6))
sns.barplot(data=top_mid_categories, x='중분류 카테고리', y='검색건수', ax=ax1, palette='Blues_r')
ax1.set_title('게스트 인기 검색 중분류 카테고리 Top 10', fontsize=16, pad=15)
ax1.set_ylabel('총 검색 건수')
ax1.set_xlabel('')
ax1.tick_params(axis='x', rotation=0)

# 레이블 추가 (천 단위 콤마 포함)
for c in ax1.containers:
    labels = [f'{int(v.get_height()):,}건' for v in c]
    ax1.bar_label(c, labels=labels, padding=3, fontsize=10, color='black')

ax1.yaxis.set_major_formatter(formatter)
plt.tight_layout()
plt.show()


# ==============================================================
# 2. 두 번째 그래프: 개별 관광지명 (교통시설 제외)
# ==============================================================
fig2, ax2 = plt.subplots(figsize=(12, 6))
sns.barplot(data=top_places, x='관광지명', y='검색건수', ax=ax2, palette='Greens_r')
ax2.set_title('게스트 인기 검색 장소 Top 10 (교통시설 제외)', fontsize=16, pad=15)
ax2.set_ylabel('총 검색 건수')
ax2.set_xlabel('관광지명')
ax2.tick_params(axis='x', rotation=45)

# 레이블 추가
for c in ax2.containers:
    labels = [f'{int(v.get_height()):,}건' for v in c]
    ax2.bar_label(c, labels=labels, padding=3, fontsize=10, color='black')

ax2.yaxis.set_major_formatter(formatter)
plt.tight_layout()
plt.show()


# ==============================================================
# 3. 세 번째 그래프: 교통시설명
# ==============================================================
fig3, ax3 = plt.subplots(figsize=(12, 6))
sns.barplot(data=top_transport, x='교통시설명', y='검색건수', ax=ax3, palette='Oranges_r')
ax3.set_title('게스트 인기 검색 교통시설 Top 10', fontsize=16, pad=15)
ax3.set_ylabel('총 검색 건수')
ax3.set_xlabel('교통시설명')
ax3.tick_params(axis='x', rotation=45)

# 레이블 추가
for c in ax3.containers:
    labels = [f'{int(v.get_height()):,}건' for v in c]
    ax3.bar_label(c, labels=labels, padding=3, fontsize=10, color='black')

ax3.yaxis.set_major_formatter(formatter)
plt.tight_layout()
plt.show()

In [ ]:
# 1. 인기 중분류 카테고리 추출 (상위 9개)
top9_mid_list = df_search_rank.groupby('중분류 카테고리')['검색건수'].sum().sort_values(ascending=False).head(9).index

# 2. 3행 3열의 서브플롯 생성 (총 9개 그래프에 딱 맞춤)
fig, axes = plt.subplots(3, 3, figsize=(20, 18))
axes = axes.flatten()

# 3. 각 중분류 카테고리별로 반복하며 그래프 그리기
for i, mid_cat in enumerate(top9_mid_list):
    # 해당 중분류 데이터만 필터링
    df_subset = df_search_rank[df_search_rank['중분류 카테고리'] == mid_cat]

    # 소분류 카테고리별 검색건수 합산 및 Top 10 추출
    top_sub = df_subset.groupby('소분류 카테고리')['검색건수'].sum().reset_index()
    top_sub = top_sub.sort_values(by='검색건수', ascending=False).head(10)

    # 막대 그래프 그리기
    sns.barplot(data=top_sub, x='검색건수', y='소분류 카테고리', ax=axes[i], palette='viridis')

    # 그래프 디자인 설정
    axes[i].set_title(f"[{mid_cat}] 내 인기 소분류 Top 10", fontsize=15, pad=10)
    axes[i].set_xlabel('총 검색 건수')
    axes[i].set_ylabel('')

    # X축 천 단위 콤마 포맷 적용
    axes[i].xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: format(int(x), ',')))

plt.tight_layout()
plt.show()

In [ ]:
# 1. 인기 중분류 카테고리 추출 (상위 9개)
top9_mid_list = df_search_rank.groupby('중분류 카테고리')['검색건수'].sum().sort_values(ascending=False).head(9).index

# 2. 3행 3열의 서브플롯 생성 (총 9칸)
fig, axes = plt.subplots(3, 3, figsize=(20, 18))
axes = axes.flatten()

# 3. 각 중분류 카테고리별로 반복하며 그래프 그리기
for i, mid_cat in enumerate(top9_mid_list):
    # 해당 중분류 데이터만 필터링
    df_subset = df_search_rank[df_search_rank['중분류 카테고리'] == mid_cat]

    # 관광지명별 검색건수 합산 및 Top 10 추출
    top_places = df_subset.groupby('관광지명')['검색건수'].sum().reset_index()
    top_places = top_places.sort_values(by='검색건수', ascending=False).head(10)

    # 막대 그래프 그리기 (y축을 관광지명으로 설정)
    sns.barplot(data=top_places, x='검색건수', y='관광지명', ax=axes[i], palette='viridis')

    # 그래프 디자인 설정
    axes[i].set_title(f"[{mid_cat}] 내 인기 관광지 Top 10", fontsize=15, pad=10)
    axes[i].set_xlabel('총 검색 건수')
    axes[i].set_ylabel('')

    # X축 천 단위 콤마 포맷 적용
    axes[i].xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: format(int(x), ',')))

plt.tight_layout()
plt.show()

# 좌표추가

In [ ]:
import pandas as pd
import requests
import time
import geopandas as gpd

BASE_DIR = os.getcwd()                                   # 현재 작업 중인 기본 폴더 경로
FOLDER_DIR = os.path.join(BASE_DIR, "병합완료")          # 가공 데이터 저장 폴더
file_path = os.path.join(FOLDER_DIR, "카테고리별_검색순위.csv")
try:
    df_search = pd.read_csv(file_path, encoding='utf-8')
except UnicodeDecodeError:
    df_search = pd.read_csv(file_path, encoding='cp949')

unique_places = df_search['관광지명'].dropna().unique()

KAKAO_API_KEY = "730b9ca38455d1c6d213945b0a29e015"
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}

address_dict = {}
x_dict = {}
y_dict = {}

for idx, place in enumerate(unique_places):
    url = f"https://dapi.kakao.com/v2/local/search/keyword.json?query={place}"

    try:
        response = requests.get(url, headers=headers)
        result = response.json()

        if result.get('documents'):
            address = result['documents'][0].get('address_name', '')
            road_address = result['documents'][0].get('road_address_name', '')
            final_address = road_address if road_address else address

            x = result['documents'][0].get('x', '')
            y = result['documents'][0].get('y', '')

            address_dict[place] = final_address
            x_dict[place] = x
            y_dict[place] = y
        else:
            address_dict[place] = "검색 결과 없음"
            x_dict[place] = None
            y_dict[place] = None

    except Exception as e:
        address_dict[place] = "에러"
        x_dict[place] = None
        y_dict[place] = None

    time.sleep(0.1)

    if (idx + 1) % 100 == 0:
        print(f"{idx + 1}개 장소 검색 완료...")

df_search['주소'] = df_search['관광지명'].map(address_dict)
df_search['X좌표 (WGS84)'] = pd.to_numeric(df_search['관광지명'].map(x_dict), errors='coerce')
df_search['Y좌표 (WGS84)'] = pd.to_numeric(df_search['관광지명'].map(y_dict), errors='coerce')

df_valid = df_search.dropna(subset=['X좌표 (WGS84)', 'Y좌표 (WGS84)']).copy()
df_invalid = df_search[df_search['X좌표 (WGS84)'].isna()].copy()

print(f"\n✅ 좌표 검색 성공 데이터 개수: {len(df_valid)}개")
if len(df_valid) > 0:
    print("   [성공 데이터 예시]:", df_valid['관광지명'].unique()[:5])

print(f"\n❌ 좌표 검색 실패 데이터 개수: {len(df_invalid)}개")
if len(df_invalid) > 0:
    print("   [실패 데이터 예시]:", df_invalid['관광지명'].unique()[:10])
print("==========================================\n")

gdf_valid = gpd.GeoDataFrame(
    df_valid,
    geometry=gpd.points_from_xy(df_valid['X좌표 (WGS84)'], df_valid['Y좌표 (WGS84)']),
    crs='EPSG:4326'
)

gdf_5179 = gdf_valid.to_crs(epsg=5179)
gdf_5179['X좌표 (GRS80TM)'] = gdf_5179.geometry.x
gdf_5179['Y좌표 (GRS80TM)'] = gdf_5179.geometry.y

df_invalid['geometry'] = None
gdf_invalid = gpd.GeoDataFrame(df_invalid, geometry='geometry', crs='EPSG:5179')

gdf_search_final = pd.concat([gdf_5179, gdf_invalid], ignore_index=True)


BASE_DIR = Path(__file__).resolve().parent

csv_output_filepath = BASE_DIR / "병합완료" / "카테고리별_검색순위_5179.csv"
gpkg_output_filepath = BASE_DIR / "병합완료" / "카테고리별_검색순위_5179.gpkg"

pd.DataFrame(gdf_search_final).to_csv(csv_output_filepath, index=False, encoding='utf-8-sig')
gdf_search_final.to_file(gpkg_output_filepath, driver='GPKG', encoding='utf-8')

In [ ]:
import pandas as pd
import requests
import time
import geopandas as gpd


# 2. 변경할 이름 딕셔너리 (논의된 내용 완벽 반영)
rename_dict = {
    # 터미널/교통시설
    '김포국제공항국내선': '김포공항 국내선',
    '서울고속버스터미널경부영동선': '서울고속버스터미널',
    '동서울종합터미널': '동서울터미널',
    '신세계센트럴시티터미널호남선': '센트럴시티터미널',
    '이랜드크루즈양화선착장': '양화선착장',
    '동력수상레저기구조정면허서울pc시험장': '조종면허 서울시험장',
    '서울마리나클럽앤요트': '서울마리나',
    '서울마리나계류장': '서울마리나',
    '더리버마리나': '더리버 마리나',
    '한강아라호': '아라호',

    # 청와대 관련 시설 (모두 '청와대'로 대통합)
    '청와대연풍문': '청와대',
    '청와대춘추관': '청와대',
    '청와대시화문': '청와대',
    '청와대대통령관저': '청와대',
    '청와대연무관': '청와대',
    '청와대춘추문': '청와대',
    '청와대영빈관': '청와대',
    '청와대대통령비서실여민2관': '청와대',
    '청와대앞길': '청와대',

    # 공원/둘레길/꽃길
    '불광천벚꽃길': '불광천',
    '방화근린공원벚꽃길': '방화근린공원',
    '매헌시민의숲벚꽃길': '매헌시민의숲',
    '양재천변': '양재천',
    '장지천벚꽃길': '장지천',
    '북한산국립공원정릉탐방안내소 공사중(2026년08월 완공예정)': '북한산국립공원 정릉탐방지원센터',
    '난지테마관광숲길': '난지한강공원',
    '당현천바닥분수대': '당현천 바닥분수',
    '남산산책로B코스': '남산공원 남산산책로',
    '남산둘레길남측순환로': '남산공원 남측순환로',
    '강서한강공원가족피크닉장': '강서한강공원 피크닉장',
    '청계산원터골입구코스': '원터골 입구',
    '수락산동막골자연휴양림건축중(2025년05월 완공예정)': '수락산자연휴양림',
    '수락산동막골자연휴양림건축중(2025년07월 완공예정)': '수락산자연휴양림',
    '올림픽공원9경스탬프투어': '올림픽공원',
    '선바위(선암)': '선바위',
    '만리재길': '만리재로',
    '흥인지문성곽공원': '흥인지문공원',

    # 체육/스포츠 시설
    '목동실내아이스링크': '목동아이스링크',
    '도봉윈터스노우랜드': '도봉 윈터스노우랜드',
    '렛츠런CCC영등포지점': '한국마사회 영등포지사',
    '렛츠런CCC도봉지점': '한국마사회 도봉지사',
    '동대문구중랑천눈썰매장': '중랑천 눈썰매장',
    '서울광장스케이트장(2024년12월20일~2025년02월09일)': '서울광장 스케이트장',
    '서울광장스케이트장(운영종료)': '서울광장 스케이트장',
    '무지개어린이눈썰매장': '어린이대공원 눈썰매장',
    '하하호호올림픽눈썰매장': '올림픽공원 눈썰매장',
    '양재천수영장겨울눈놀이터': '양재천 수영장',
    '다락원체육공원실내배드민턴장': '다락원체육공원 배드민턴장',
    '탄천파크골프장A코스': '탄천파크골프장',
    '탄천파크골프장C코스': '탄천파크골프장',
    '강남탄천파크골프장관리실': '탄천파크골프장',
    '강남탄천파크골프장A코스': '탄천파크골프장',
    '강남탄천파크골프장C코스': '탄천파크골프장',
    '한내파크골프장': '한내천 파크골프장',
    '우장산인조잔디구장': '우장산축구장',
    '안양천야구장1구장': '안양천야구장',
    '양평누리체육공원야구장': '양평유수지야구장',
    '스페셜스포츠센터': '스페셜올림픽코리아',
    '한강시민공원잠원지구수영장': '잠원한강공원 수영장',

    # 박물관/미술관/공연장
    '서울아레나건축중(2027년03월 준공예정)': '서울아레나',
    '서울혁신파크(폐쇄)': '서울혁신파크',
    '국립한글박물관휴관중(2025년10월 개관예정)': '국립한글박물관',
    '서서울미술관건축중(2025년11월 개관예정)': '서서울미술관',
    '당인리문화창작발전소건축중(2026년 개관예정)': '당인리문화창작발전소',
    '민주화운동기념관휴관중(2025년06월 개관예정)': '민주화운동기념관',
    '국립한글박물관휴관중(2025년10월09일 개관예정)': '국립한글박물관',
    '국립한글박물관휴관중(2028년10월 개관예정)': '국립한글박물관',
    '화정박물관휴업중(2026년상반기오픈예정)': '화정박물관',
    'GS아트센터(2025년04월24일 개관예정)': 'GS아트센터',
    '서울극장폐점': '서울극장',

    # 쇼핑/시장
    '서울경동약령시장': '서울약령시장',
    '강동아이파크더리버공사중(2025년04월17일오픈)': '고덕아이파크디어반',
    '롯데영플라자명동점 공사중(2026년오픈예정)': '롯데영플라자 본점',
    '양곡도매시장신시장 건축중(2025년12월 완공예정)': '양재동 양곡도매시장',
    '가락동농수산물도매시장직판시장': '가락시장',
    '뽀로로파크롯데마트맥스금천점': '롯데마트맥스 금천점 뽀로로파크',

    # 식당/카페
    '긴자올림픽점 [일식]': '긴자 올림픽점',
    '해목논현점 [일식]': '해목 논현점',
    '홍성원마곡점 [중식]': '홍성원 마곡점',
    '만복기사식당[한식]': '만복기사식당',
    '오미가압구정점 [한식]': '오미가 압구정점',
    '진진본관 [중식]': '진진 본관',
    '서라벌한정식서초점 [한식]': '서라벌한정식 서초점',
    '마코토청담점 [일식]': '마코토 청담점',
    '우래옥본점 공사중(2025년09월오픈예정)': '우래옥 본점',
    '또순이네공사중(2025년08월24일오픈예정)': '또순이네',
    '만애옥한정식송파가락시장본점 [한식]': '만애옥',
    '한국의집휴업중(2026년01월오픈예정) [한식]': '한국의집',
    '더라운드한남점 [중식]': '더라운드 한남점',

    # 호텔/숙박
    '웨스틴서울파르나스호텔공사중(2025년09월오픈예정)': '웨스틴 서울 파르나스',
    '웨스틴서울파르나스호텔공사중(2025년09월15일오픈예정)': '웨스틴 서울 파르나스',
    '밀레니엄서울힐튼호텔폐업': '밀레니엄힐튼서울',
    '카파스호텔공사중(2025년03월 오픈예정)': '카파스호텔',
    '나이아가라호텔공사중(2025년03월04일 오픈예정)': '나이아가라호텔',
    '레이크호텔공사중(2025년08월10일오픈예정)': '레이크호텔',
    '론스타호텔공사중(2025년09월 오픈예정)': '론스타호텔',
    'VL르웨스트건축중(2025년10월 준공예정)': 'VL르웨스트',
    '시그니엘서울호텔연회.메인': '시그니엘 서울',

    # 기타
    '토모gym': '토모짐',
    '바디사이언스베이스볼퍼포먼스': '바디사이언스 베이스볼',
    '당고개실내배드민턴장': '당고개 배드민턴장',
    '롤러클럽은평충암점': '롤러클럽 충암점',
    '서울볼더스선유점': '서울볼더스',
    '아이스온발산센터': '아이스온',
    '클라임잇 클라이밍구로점': '클라임잇 구로',
    '온플릭클라이밍짐천호점': '온플릭 클라이밍',
    '존프랭클 샤카주짓수압구정아카데미': '샤카주짓수',
    '히트배팅존&레전드야구존강서화곡점': '레전드야구존 화곡점',
    '프라우드바디자이로토닉여의도점': '프라우드바디',
    '헬로방방마포아현점': '헬로방방 아현점',
    '챔피언1250XHDC아이파크몰용산점': '챔피언1250 아이파크몰용산점',
    '챔피언1250강동아이파크더리버몰점': '챔피언1250 강동',
    '현인릉관리사무소': '헌인릉',
    '양도공묘역': '양도공 이첩 묘역',
    '노원구 중계동 104번지마을': '백사마을',
    '대한불교천태관문사': '관문사'
}

# 3. 딕셔너리에 작성한 이름으로 임시 검색용 컬럼 생성
gdf_search_final['검색용_관광지명'] = gdf_search_final['관광지명'].replace(rename_dict)

# 4. 좌표가 비어있는 데이터 중 검색용 이름이 바뀐 항목만 재검색 대상으로 추출
retry_places = gdf_search_final[gdf_search_final['X좌표 (WGS84)'].isna()]['검색용_관광지명'].dropna().unique()
print(f"🔄 총 {len(retry_places)}개의 장소 재검색을 시작합니다...\n")

KAKAO_API_KEY = "730b9ca38455d1c6d213945b0a29e015"
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}

retry_x_dict = {}
retry_y_dict = {}
retry_addr_dict = {}

for idx, place in enumerate(retry_places):
    url = f"https://dapi.kakao.com/v2/local/search/keyword.json?query={place}"
    try:
        response = requests.get(url, headers=headers)
        result = response.json()

        if result.get('documents'):
            retry_x_dict[place] = float(result['documents'][0].get('x', 0))
            retry_y_dict[place] = float(result['documents'][0].get('y', 0))
            addr = result['documents'][0].get('road_address_name', '')
            if not addr:
                addr = result['documents'][0].get('address_name', '')
            retry_addr_dict[place] = addr
        else:
            retry_x_dict[place] = None
            retry_y_dict[place] = None
            retry_addr_dict[place] = None
    except Exception:
        retry_x_dict[place] = None
        retry_y_dict[place] = None
        retry_addr_dict[place] = None

    time.sleep(0.1)
    if (idx + 1) % 20 == 0:
        print(f"{idx + 1}개 재검색 완료...")

# 5. 재검색에 성공한 좌표와 주소를 기존 데이터의 빈칸에 채워넣기
for place in retry_places:
    if retry_x_dict.get(place):
        mask = (gdf_search_final['검색용_관광지명'] == place) & (gdf_search_final['X좌표 (WGS84)'].isna())
        gdf_search_final.loc[mask, 'X좌표 (WGS84)'] = retry_x_dict[place]
        gdf_search_final.loc[mask, 'Y좌표 (WGS84)'] = retry_y_dict[place]
        gdf_search_final.loc[mask, '주소'] = retry_addr_dict[place]

# 6. 좌표가 모두 업데이트된 상태에서 WGS84 -> 5179 변환 진행
df_valid = gdf_search_final.dropna(subset=['X좌표 (WGS84)', 'Y좌표 (WGS84)']).copy()
df_invalid = gdf_search_final[gdf_search_final['X좌표 (WGS84)'].isna()].copy()

print(f"\n✅ 재검색 완료! 전체 성공 데이터 개수: {len(df_valid)}개")
print(f"❌ 여전히 실패한 데이터 개수: {len(df_invalid)}개")

# (1) WGS84 기준 GeoDataFrame 생성
gdf_valid = gpd.GeoDataFrame(
    df_valid,
    geometry=gpd.points_from_xy(df_valid['X좌표 (WGS84)'], df_valid['Y좌표 (WGS84)']),
    crs='EPSG:4326'
)

# (2) 5179로 좌표 변환
gdf_5179 = gdf_valid.to_crs(epsg=5179)
gdf_5179['X좌표 (GRS80TM)'] = gdf_5179.geometry.x
gdf_5179['Y좌표 (GRS80TM)'] = gdf_5179.geometry.y

# (3) 실패 데이터 빈 geometry로 맞추기
df_invalid['geometry'] = None
gdf_invalid = gpd.GeoDataFrame(df_invalid, geometry='geometry', crs='EPSG:5179')

# (4) 성공/실패 다시 하나로 병합 후 임시 컬럼 삭제
gdf_search_final = pd.concat([gdf_5179, gdf_invalid], ignore_index=True)
gdf_search_final = gdf_search_final.drop(columns=['검색용_관광지명'], errors='ignore')


In [ ]:
# X좌표가 여전히 NaN인 데이터만 필터링
still_failed = gdf_search_final[gdf_search_final['X좌표 (WGS84)'].isna()]

# 고유 관광지명을 가나다순으로 정렬해서 추출
failed_list = sorted(still_failed['관광지명'].unique())

print(f"❌ 여전히 검색에 실패한 고유 관광지 수: {len(failed_list)}개")
print("-" * 50)

# 리스트 형태로 출력 (복사해서 확인하기 좋게)
for place in failed_list:
    print(f"'{place}',")

In [ ]:
import pandas as pd
import requests
import time
import geopandas as gpd
from shapely.geometry import Point


# 2. 카카오맵 검색 최적화 딕셔너리
rename_dict = {
    # [교통 및 선착장]
    '김포국제공항국내선': '김포공항 국내선', '서울고속버스터미널경부영동선': '서울고속버스터미널',
    '동서울종합터미널': '동서울터미널', '신세계센트럴시티터미널호남선': '센트럴시티터미널',
    '이랜드크루즈양화선착장': '양화선착장', '동력수상레저기구조정면허서울pc시험장': '조종면허 서울시험장',
    '서울마리나클럽앤요트': '서울마리나', '서울마리나계류장': '서울마리나', '더리버마리나': '더리버 마리나',
    '한강아라호': '아라호', '한국윈드서핑교육원': '한국윈드서핑교육원',

    # [청와대 대통합]
    '청와대연풍문': '청와대', '청와대춘추관': '청와대', '청와대시화문': '청와대',
    '청와대대통령관저': '청와대', '청와대연무관': '청와대', '청와대춘추문': '청와대',
    '청와대영빈관': '청와대', '청와대대통령비서실여민2관': '청와대', '청와대앞길': '청와대',

    # [공원/둘레길/명소]
    '서울빛초롱축제(청계천)': '청계광장', '불광천벚꽃길': '불광천', '방화근린공원벚꽃길': '방화근린공원',
    '매헌시민의숲벚꽃길': '매헌시민의숲', '양재천변': '양재천', '장지천벚꽃길': '장지천',
    '난지테마관광숲길': '난지한강공원', '남산산책로B코스': '남산공원 남산산책로',
    '남산둘레길남측순환로': '남산공원 남측순환로', '흥인지문성곽공원': '흥인지문공원',
    '석촌고분공원': '서울 석촌동 고분군', '백제초기적석총': '석촌동 고분군',
    '불암산철쭉제': '불암산 나비정원', '성수대교사고희생자위령비': '성수대교 희생자 위령비',
    '만리재길': '만리재로', '선바위(선암)': '선바위', '방이맛골': '방이맛골',

    # [스포츠/레저/체육]
    '목동실내아이스링크': '목동아이스링크', '도봉윈터스노우랜드': '도봉 윈터스노우랜드',
    '렛츠런CCC영등포지점': '한국마사회 영등포지사', '렛츠런CCC도봉지점': '한국마사회 도봉지사',
    '양신스포츠아카데미': '양준혁 야구재단', '남현희인터내셔널펜싱아카데미': '남현희 인터내셔널 펜싱아카데미',
    '100마일베이스볼': '100마일 야구아카데미', 'JT야구레슨장': 'JT 야구레슨장',
    'RBT베이스볼아카데미': 'RBT 베이스볼 아카데미', '탄천파크골프장A코스': '탄천파크골프장',
    '양평누리체육공원야구장': '양평유수지야구장', '챔피언1250XHDC아이파크몰용산점': '챔피언1250X 아이파크몰 용산점',

    # [프랜차이즈/식당/카페 정제]
    'VIPS프리미어목동41타워점': '빕스 목동41타워점', 'VIPS프리미어미아점': '빕스 미아점',
    'VIPS프리미어반포역점': '빕스 반포역점', 'VIPS현대시티아울렛가든파이브점': '빕스 가든파이브점',
    '63뷔페파빌리온HDC아이파크몰용산점': '63뷔페 파빌리온', 'SEA LIFE 코엑스아쿠아리움': '코엑스 아쿠아리움',
    '하이디라오현대시티아울렛가산점': '하이디라오 가산점', '청와옥사당직영점': '청와옥 사당점',
    '또순이네집마곡동점': '또순이네 마곡점', '또순이네공사중(2025년08월24일오픈예정)': '또순이네',
    '대도식당삼성직영점': '대도식당 삼성점', '덕후선생청담점': '덕후선생 청담점',

    # [호텔/숙박]
    '로사나부띠끄호텔': '로사나 호텔', '호텔스카이파크동대문1호점': '호텔 스카이파크 동대문 1호점',
    '카파쓰관광호텔': '카파스호텔', '카파스호텔공사중(2025년03월 오픈예정)': '카파스호텔',
    '밀레니엄서울힐튼호텔폐업': '밀레니엄 힐튼 서울', '브라운도트호텔신촌점': '브라운도트 신촌점'}

# 3. 딕셔너리 적용 및 재검색 대상 추출
gdf_search_final ['검색용_관광지명'] = gdf_search_final ['관광지명'].replace(rename_dict)
mask_retry = gdf_search_final ['X좌표 (WGS84)'].isna() & (gdf_search_final['검색용_관광지명'] != gdf_search_final ['관광지명'])
retry_places = gdf_search_final [mask_retry]['검색용_관광지명'].unique()

print(f"🔄 총 {len(retry_places)}개의 고유 장소를 재검색합니다...")

# 4. 카카오 API 재검색 실행
KAKAO_API_KEY = "730b9ca38455d1c6d213945b0a29e015"
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}

retry_results = {}
for idx, place in enumerate(retry_places):
    url = f"https://dapi.kakao.com/v2/local/search/keyword.json?query={place}"
    try:
        res = requests.get(url, headers=headers).json()
        if res.get('documents'):
            doc = res['documents'][0]
            retry_results[place] = {
                'x': float(doc['x']), 'y': float(doc['y']),
                'addr': doc.get('road_address_name') or doc.get('address_name')
            }
    except: pass
    time.sleep(0.1)
    if (idx + 1) % 20 == 0: print(f"{idx + 1}개 완료...")

# 5. 재검색 결과 업데이트
for place, data in retry_results.items():
    update_mask = (gdf_search_final ['검색용_관광지명'] == place) & (gdf_search_final ['X좌표 (WGS84)'].isna())
    gdf_search_final .loc[update_mask, ['X좌표 (WGS84)', 'Y좌표 (WGS84)', '주소']] = [data['x'], data['y'], data['addr']]

# 6. 유효 데이터 추출 및 5179 좌표 변환
df_clean = gdf_search_final.dropna(subset=['X좌표 (WGS84)']).copy()
gdf_clean = gpd.GeoDataFrame(
    df_clean,
    geometry=gpd.points_from_xy(df_clean['X좌표 (WGS84)'], df_clean['Y좌표 (WGS84)']),
    crs='EPSG:4326'
)

gdf_5179 = gdf_clean.to_crs(epsg=5179)
gdf_5179['X좌표 (GRS80TM)'] = gdf_5179.geometry.x
gdf_5179['Y좌표 (GRS80TM)'] = gdf_5179.geometry.y

# X, Y 좌표 컬럼을 이용하여 geometry 컬럼 생성 (또는 업데이트)
gdf_5179['geometry'] = gpd.points_from_xy(gdf_5179['X좌표 (GRS80TM)'], gdf_5179['Y좌표 (GRS80TM)'])

# 만약 CRS(좌표계) 정보가 유실되었다면 다시 설정해주는 것이 좋습니다
gdf_5179.set_crs(epsg=5179, inplace=True)

# 1. 경로 설정 (현재 작업 폴더 기준)
BASE_DIR = os.getcwd()                                              # 현재 작업 중인 기본 폴더 경로
FOLDER_DIR = os.path.join(BASE_DIR, "병합완료")                     # 가공 데이터 저장 폴더

# 2. 출력 파일 경로 설정 (gdf_search_final 저장용)
output_csv = os.path.join(FOLDER_DIR, "카테고리별_검색순위_5179_최종_Clean.csv")
output_gpkg = os.path.join(FOLDER_DIR, "카테고리별_검색순위_5179_최종_Clean.gpkg")

# 불필요한 보조 컬럼 제거 후 저장
df_to_save = pd.DataFrame(gdf_5179).drop(columns=['검색용_관광지명'], errors='ignore')
df_to_save.to_csv(output_csv, index=False, encoding='utf-8-sig')
gdf_5179.drop(columns=['검색용_관광지명'], errors='ignore').to_file(output_gpkg, driver='GPKG', encoding='utf-8')


## 좌표수정

In [ ]:
import os
import pandas as pd

BASE_DIR = os.getcwd()                     # 현재 작업 중인 기본 폴더 경로 (/Users/yjh)
FOLDER_DIR = os.path.join(BASE_DIR, "병합완료")  # 가공 데이터 저장 폴더

# 불러올 파일 이름 지정
DATA_DIR = os.path.join(FOLDER_DIR, "카테고리별_검색순위_5179_최종_Clean.csv")

# 데이터프레임으로 불러오기 (데이터 성격에 맞춰 변수명을 df_search_rank로 지정)
df_clean = pd.read_csv(DATA_DIR, index_col=0)

In [ ]:
import requests
import time

# 1. 카카오 REST API 키 입력 (기존에 사용하시던 키를 넣어주세요)
KAKAO_API_KEY = "730b9ca38455d1c6d213945b0a29e015"

def get_correct_address(gu, place_name):
    # 검색어를 "서울 + 자치구 + 관광지명"으로 구체화하여 정확도 향상
    search_query = f"서울 {gu} {place_name}"
    url = f"https://dapi.kakao.com/v2/local/search/keyword.json?query={search_query}"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}

    try:
        response = requests.get(url, headers=headers)
        result = response.json()

        if result.get('documents'):
            # 첫 번째 검색 결과의 도로명 주소(없으면 지번 주소) 가져오기
            address = result['documents'][0].get('road_address_name')
            if not address:
                address = result['documents'][0].get('address_name')
            return address
        else:
            return None # 검색 결과가 아예 없는 경우
    except Exception as e:
        return None

# 2. 344건에 대해서만 새 주소 찾기
print("API 검색을 시작합니다. (약 30초~1분 소요 예상)")
new_addresses = {}

for idx, row in df_mismatch_unique.iterrows():
    gu = row['자치구']
    place = row['관광지명']

    # API 호출
    new_addr = get_correct_address(gu, place)
    new_addresses[place] = new_addr

    # API 호출 제한 방지를 위해 0.1초 대기
    time.sleep(0.1)

print("새로운 주소 검색이 완료되었습니다!")

# 3. 찾은 새 주소를 원본 데이터(df_clean)에 덮어쓰기
# '관광지명'을 기준으로 매핑할 임시 컬럼 생성
df_clean['새_주소'] = df_clean['관광지명'].map(new_addresses)

In [ ]:
# 4. '새_주소'에 값이 있는 경우(API 검색 성공)에만 기존 '주소'를 덮어쓰고,
# 값이 없는 경우(API 검색 실패, NaN)에는 기존 '주소'를 그대로 유지합니다.
df_clean['주소'] = df_clean['새_주소'].fillna(df_clean['주소'])

# 5. 업데이트가 끝났으니 임시로 만든 '새_주소' 컬럼은 삭제하여 깔끔하게 정리합니다.
df_clean.drop(columns=['새_주소'], inplace=True)

print("기존 '주소' 컬럼에 새 주소 업데이트 및 정리가 완료되었습니다!")

In [ ]:
# 상대경로 지정 (현재 작업 중인 폴더 아래의 '병합완료' 폴더)
save_path = "./병합완료/카테고리별_검색순위_5179_최종_Clean.csv"

# CSV 파일로 저장
# index=False : 데이터프레임의 숫자 인덱스를 파일에 따로 저장하지 않음
# encoding='utf-8-sig' : 엑셀에서 열었을 때 한글이 깨지는 것을 방지
df_clean.to_csv(save_path, index=False, encoding="utf-8-sig")

print(f"파일이 성공적으로 저장되었습니다: {save_path}")

# 키워드_관광지

In [ ]:
BASE_DIR = os.getcwd()                     # 현재 작업 중인 기본 폴더 경로
FOLDER_DIR = os.path.join(BASE_DIR, "병합완료")  # 가공 데이터 저장 폴더

# 수정된 부분: 가져올 csv 파일명 변경
DATA_DIR = os.path.join(FOLDER_DIR, "카테고리별_검색순위_5179_최종_Clean.csv")
df_search  = pd.read_csv(DATA_DIR, encoding='utf-8')

In [ ]:
df_search.head()

In [ ]:
df_search.columns

In [ ]:
# 1. 제거할 컬럼 리스트
cols_to_drop = ['Unnamed: 0', '순위', '광역시/도', '관광지ID']

# 2. 컬럼 제거 적용 (기존 df를 df_search로 받아서 처리하거나, 이미 df_search라면 그대로 사용)
df_search = df_search.drop(columns=cols_to_drop, errors='ignore')

# 3. 결과 확인
print(df_search.columns)

In [ ]:
# 1. 유지하고 싶은 컬럼들을 index 리스트에 추가
df_pivot = df_search.pivot_table(
    index=['중분류 카테고리', '소분류 카테고리', '관광지명'],

    columns='월',
    values='검색건수',
    aggfunc='sum'
).fillna(0)

# 2. 컬럼명 정리 (예: 1 -> 1월)
df_pivot.columns = [f'{int(col)}월' for col in df_pivot.columns]

# 3. 인덱스 재정렬 (자치구 순 -> 관광지명 순 등으로 계층적 정렬)
df_pivot = df_pivot.sort_index()

In [ ]:
df_pivot.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 맥(Mac) 환경 한글 폰트 설정
plt.rc('font', family='AppleGothic')
plt.rcParams['axes.unicode_minus'] = False

# ==========================================
# 1. Top 10 관광지 추출
# ==========================================
# 기존 피벗 테이블 복사
df_pivot_top = df_pivot.copy()

# 각 행(관광지)별 1~12월 검색건수 총합 계산하여 새로운 컬럼 생성
df_pivot_top['총합'] = df_pivot_top.sum(axis=1)

# '총합' 기준으로 내림차순 정렬 후 상위 10개만 추출
df_top10 = df_pivot_top.sort_values(by='총합', ascending=False).head(10)

# 시각화할 때는 '총합' 컬럼이 필요 없으므로 제거
df_top10 = df_top10.drop(columns=['총합'])


# ==========================================
# 2. 데이터 변환 (행과 열 바꾸기)
# ==========================================
# 전치(.T)하여 X축을 '월'로 설정
df_trend_top10 = df_top10.T

# 다중 인덱스에서 '관광지명'만 범례로 사용하기 위해 컬럼명 정리
if isinstance(df_trend_top10.columns, pd.MultiIndex):
    df_trend_top10.columns = df_trend_top10.columns.get_level_values('관광지명')


# ==========================================
# 3. 시각화 (선 그래프)
# ==========================================
plt.figure(figsize=(14, 8))

# Top 10 관광지에 대한 선 그래프 (팔레트를 tab10으로 주어 10개 색상을 명확하게 구분)
sns.lineplot(data=df_trend_top10, markers=True, dashes=False, palette='tab10', linewidth=2)

plt.title('연간 검색건수 Top 10 관광지 월간 추이 변화', fontsize=16)
plt.xlabel('월', fontsize=12)
plt.ylabel('검색건수', fontsize=12)

# 범례를 우측 바깥으로 배치
plt.legend(title='관광지명 (Top 10)', bbox_to_anchor=(1.02, 1), loc='upper left')

# 수치 비교를 돕는 격자(그리드) 추가
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# 0. '소분류 카테고리'가 '교통시설'이 아닌 데이터만 필터링 (핵심 추가)
df_filtered = df_search[df_search['소분류 카테고리'] != '교통시설']

# 1. 필터링된 데이터에서 중분류와 월별로 그룹화하여 검색건수가 최대인 인덱스 추출
idx = df_filtered.groupby(['중분류 카테고리', '월'])['검색건수'].idxmax()

# 2. 해당 인덱스의 데이터만 가져오기 (관광지명 포함)
df_monthly_top = df_filtered.loc[idx, ['중분류 카테고리', '월', '관광지명', '검색건수']]

# 3. 시각화를 위해 피벗 (행: 중분류, 열: 월, 값: 관광지명)
top_names_pivot = df_monthly_top.pivot(index='중분류 카테고리', columns='월', values='관광지명')

In [ ]:

# 이름이 길면 앞 6글자만 보여주고 '..'을 붙이는 함수
def shorten_name(name, max_len=6):
    if len(str(name)) > max_len:
        return str(name)[:max_len] + '..'
    return str(name)

# 3. 피벗 생성 (히트맵 색상용 값)
top_values_pivot = df_monthly_top.pivot(index='중분류 카테고리', columns='월', values='검색건수')

# 4. 피벗 생성 (히트맵 텍스트 표시용 - 짧은 이름 적용)
top_names_pivot_short = df_monthly_top.pivot(index='중분류 카테고리', columns='월', values='관광지명').map(shorten_name)


# --- 시각화 ---
plt.figure(figsize=(16, 10)) # 가로를 조금 더 넓게 설정

sns.heatmap(
    top_values_pivot,
    annot=top_names_pivot_short.values, # 짧게 줄인 이름을 텍스트로 사용
    fmt="",
    cmap="YlGnBu",
    # 텍스트 스타일 조절 (글자 크기 줄임)
    annot_kws={'size': 9, 'weight': 'bold', 'color': 'black'}
)

plt.title('카테고리별 월별 상위 검색 장소 히트맵', fontsize=15)
plt.xlabel('월', fontsize=12)
plt.ylabel('중분류 카테고리', fontsize=12)
plt.xticks(rotation=0) # X축 레이블 회전 방지
plt.tight_layout()
plt.show()

In [ ]:
# 0. '소분류 카테고리'가 '교통시설'이 아닌 데이터만 필터링
df_filtered = df_search[df_search['소분류 카테고리'] != '교통시설']

# 1. 필터링된 데이터에서 '자치구'와 월별로 그룹화하여 검색건수가 최대인 인덱스 추출
idx = df_filtered.groupby(['자치구', '월'])['검색건수'].idxmax()

# 2. 해당 인덱스의 데이터만 가져오기 (변수명 변경: df_monthly_top -> df_gu_monthly_top)
df_gu_monthly_top = df_filtered.loc[idx, ['자치구', '월', '관광지명', '검색건수']]


# ==========================================
# 2. 시각화를 위한 피벗 테이블 생성
# ==========================================

# 이름이 길면 앞 6글자만 보여주고 '..'을 붙이는 함수
def shorten_name(name, max_len=6):
    if len(str(name)) > max_len:
        return str(name)[:max_len] + '..'
    return str(name)

# 3. 피벗 생성 (히트맵 색상용 값 / 변수명 변경: top_values_pivot -> gu_values_pivot)
gu_values_pivot = df_gu_monthly_top.pivot(index='자치구', columns='월', values='검색건수')

# 4. 피벗 생성 (히트맵 텍스트용 / 변수명 변경: top_names_pivot_short -> gu_names_pivot_short)
gu_names_pivot_short = df_gu_monthly_top.pivot(index='자치구', columns='월', values='관광지명').map(shorten_name)


# ==========================================
# 3. 시각화 (히트맵)
# ==========================================

plt.figure(figsize=(16, 10)) # 가로를 조금 더 넓게 설정

sns.heatmap(
    gu_values_pivot,
    annot=gu_names_pivot_short.values, # 짧게 줄인 이름을 텍스트로 사용
    fmt="",
    cmap="YlGnBu",
    # 텍스트 스타일 조절 (글자 크기 줄임)
    annot_kws={'size': 9, 'weight': 'bold', 'color': 'black'}
)

plt.title('자치구별 월별 상위 검색 장소 히트맵', fontsize=15)
plt.xlabel('월', fontsize=12)
plt.ylabel('자치구', fontsize=12)
plt.xticks(rotation=0) # X축 레이블 회전 방지
plt.tight_layout()
plt.show()

In [ ]:
# 1. 정렬: 카테고리 및 월별로 '검색건수'가 가장 많은 순(내림차순)으로 정렬
df_sorted = df_search.sort_values(by=['중분류 카테고리', '월', '검색건수'], ascending=[True, True, False])

# 2. Top 5 추출: 각 카테고리/월 그룹에서 상위 5개만 뽑아오기
df_top5 = df_sorted.groupby(['중분류 카테고리', '월']).head(5).copy()

# 3. 순위 매기기 (1~5위)
df_top5['순위'] = df_top5.groupby(['중분류 카테고리', '월']).cumcount() + 1

# 4. 텍스트 만들기 (예: "1. 남산서울타워")
# 표 형태는 가로 너비가 넓어지므로 굳이 글자를 자르지 않아도 잘 보입니다.
df_top5['표시텍스트'] = df_top5['순위'].astype(str) + ". " + df_top5['관광지명'].astype(str)

# 5. 같은 월, 같은 카테고리의 5개 데이터를 줄바꿈(\n)으로 묶어주기
df_text_combined = df_top5.groupby(['중분류 카테고리', '월'])['표시텍스트'].apply(lambda x: '\n'.join(x)).reset_index()

# 6. 피벗 테이블 생성 (행: 중분류, 열: 1~12월)
table_view = df_text_combined.pivot(index='중분류 카테고리', columns='월', values='표시텍스트')

# 7. 컬럼명 깔끔하게 변경 및 결측치(데이터 없는 달) 빈칸 처리
table_view.columns = [f'{int(col)}월' for col in table_view.columns]
table_view = table_view.fillna('')

# 8. 🌟 핵심 팁: 주피터 노트북에서 줄바꿈(\n)이 표에 예쁘게 적용되도록 스타일 설정 후 출력
styled_table = table_view.style.set_properties(**{
    'white-space': 'pre-wrap',  # 줄바꿈 적용
    'text-align': 'left'        # 텍스트 왼쪽 정렬 (보기 편하게)
})

# 화면에 출력
display(styled_table)

In [ ]:
# 1. 정렬: 카테고리 무시하고 '자치구'와 '월'별 검색건수 내림차순 정렬
df_sorted = df_search.sort_values(by=['자치구', '월', '검색건수'],
                                  ascending=[True, True, False])

# 2. Top 5 추출: 각 자치구의 월별 상위 5개만 뽑아오기
df_top5 = df_sorted.groupby(['자치구', '월']).head(5).copy()

# 3. 순위 매기기 (1~5위)
df_top5['순위'] = df_top5.groupby(['자치구', '월']).cumcount() + 1

# 4. 텍스트 만들기 (예: "1. 남산서울타워")
df_top5['표시텍스트'] = df_top5['순위'].astype(str) + ". " + df_top5['관광지명'].astype(str)

# 5. 같은 자치구, 같은 월의 5개 데이터를 줄바꿈(\n)으로 묶어주기
df_text_combined = df_top5.groupby(['자치구', '월'])['표시텍스트'].apply(lambda x: '\n'.join(x)).reset_index()

# 6. 피벗 테이블 생성 (행: 자치구, 열: 1~12월)
table_view = df_text_combined.pivot(index='자치구', columns='월', values='표시텍스트')

# 7. 컬럼명 깔끔하게 변경 및 결측치(데이터 없는 달) 빈칸 처리
table_view.columns = [f'{int(col)}월' for col in table_view.columns]
table_view = table_view.fillna('')

# 8. 스타일 설정 후 출력 (줄바꿈 및 정렬 적용)
styled_table = table_view.style.set_properties(**{
    'white-space': 'pre-wrap',  # 줄바꿈(\n)이 표에서 작동하도록 설정
    'text-align': 'left',       # 읽기 편하게 왼쪽 정렬
    'vertical-align': 'top'     # 텍스트가 칸의 위쪽부터 시작되도록 설정
})

# 화면에 출력
display(styled_table)

In [ ]:
# 1. 자치구가 주소에 포함되지 않는 행 필터링 (불일치 마스크)
mismatch_mask = df_clean.apply(lambda row: str(row['자치구']) not in str(row['주소']), axis=1)

# 2. 불일치하는 행들만 따로 추출
df_mismatch_all = df_clean[mismatch_mask]

# 3. '관광지명'을 기준으로 중복 제거 (첫 번째 데이터만 남김)
# 추출할 컬럼: 관광지명, 자치구, 주소
df_mismatch_unique = df_mismatch_all[['관광지명', '자치구', '주소']].drop_duplicates(subset=['관광지명'])

display(df_mismatch_unique)

In [ ]:
#행정동 SHP 파일 불러오기
base_dir = os.getcwd()
shp_path = os.path.join(base_dir, '병합완료', '서울시 상권분석서비스(영역-행정동)', '서울시 상권분석서비스(영역-행정동).shp')
gdf_dong = gpd.read_file(shp_path, encoding='cp949')

# 좌표계 자동 변환 (행정동 지도를 WGS84로 맞춤)
minx, miny, maxx, maxy = gdf_dong.total_bounds

if np.isnan(minx):
    print("🚨 [경고] SHP 파일에 지형 데이터가 없습니다! 폴더에 .shx 파일이 꼭 같이 있어야 합니다.")
else:
    if gdf_dong.crs is None:
        if maxx < 200:
            gdf_dong.set_crs(epsg=4326, inplace=True)
        elif maxy > 1000000:
            gdf_dong.set_crs(epsg=5179, inplace=True)
        else:
            gdf_dong.set_crs(epsg=5181, inplace=True)

    if gdf_dong.crs.to_epsg() != 4326:
        gdf_dong = gdf_dong.to_crs(epsg=4326)

# 3. 데이터 전처리 및 전체 기간 검색건수 합치기
df_search['X좌표 (WGS84)'] = df_search['X좌표 (WGS84)'].astype(float)
df_search['Y좌표 (WGS84)'] = df_search['Y좌표 (WGS84)'].astype(float)
df_search['검색건수'] = df_search['검색건수'].astype(float)

# 관광지별 총 검색건수 합산
df_total = df_search.groupby(['관광지명', 'X좌표 (WGS84)', 'Y좌표 (WGS84)'])['검색건수'].sum().reset_index()

# 지리 데이터(Point) 생성
gdf_points = gpd.GeoDataFrame(
    df_total,
    geometry=gpd.points_from_xy(df_total['X좌표 (WGS84)'], df_total['Y좌표 (WGS84)']),
    crs="EPSG:4326"
)

# 4. 시각화 (도화지 큼직하게 16x12)
fig, ax = plt.subplots(figsize=(16, 12))

# 행정동 바탕 지도 그리기
gdf_dong.plot(ax=ax, color='gainsboro', edgecolor='black', linewidth=1)

# 빨간 점 찍기
if not gdf_points.empty:
    sizes = gdf_points['검색건수'] / 500  # 점 크기 조절 (필요시 1000, 2000 등으로 변경)
    gdf_points.plot(ax=ax, color='red', alpha=0.5, markersize=sizes)

# ---------------------------------------------------------
# 🌟 핵심 추가: 자치구 이름 텍스트 표시하기
# 각 자치구별 관광지 좌표의 중간값(Median)을 구해서 그 위치에 이름을 씁니다.
gu_centers = df_search.groupby('자치구')[['X좌표 (WGS84)', 'Y좌표 (WGS84)']].median().reset_index()

for idx, row in gu_centers.iterrows():
    ax.annotate(row['자치구'],
                xy=(row['X좌표 (WGS84)'], row['Y좌표 (WGS84)']),
                fontsize=11,
                fontweight='bold',
                color='black',
                ha='center', va='center',
                # 글씨가 지도 선이나 빨간 점에 가려지지 않도록 반투명 흰색 배경 추가
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='none'))
# ---------------------------------------------------------

# 화면을 서울시 지도 크기에 꽉 차게 고정
minx, miny, maxx, maxy = gdf_dong.total_bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

ax.set_title('서울시 관광지 전체 검색 분포 및 자치구 현황', fontsize=22, fontweight='bold', pad=15)
ax.axis('off')

plt.tight_layout()
plt.show()